# GPU & CUDA Verification

In [ ]:
import torch
import platform

print("Python environment:", platform.python_version())
print("PyTorch version:", torch.__version__)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU error. Colab Runtime system check and verification")

gpu_name = torch.cuda.get_device_name(0)

gpu_memory = (torch.cuda.get_device_properties(0).total_memory/ (1024 ** 3))

print("\n--- GPU Information ---")
print("GPU:", gpu_name)
print(f"GPU Memory: {gpu_memory:.2f} GB")
print("CUDA Version:", torch.version.cuda)
print("BF16 Supported:", torch.cuda.is_bf16_supported())


# Install Dependencies

In [ ]:
%pip install -q \
    "transformers" \
    "peft" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "datasets==4.7.0" \
    "optuna" \
    "huggingface_hub" \
    "scikit-learn" \
    "pandas==2.2.3" \
    "numpy==2.2.6" \
    "pyarrow==23.0.1" \
    "sentencepiece" \
    "protobuf"

In [ ]:
import importlib.metadata as metadata

packages = [
    "transformers",
    "peft",
    "trl",
    "accelerate",
    "bitsandbytes",
    "datasets",
    "optuna",
    "huggingface_hub",
    "scikit-learn",
    "pandas",
    "numpy",
    "pyarrow",
    "sentencepiece",
    "protobuf"
]

for package in packages:
    print(f"{package}: {metadata.version(package)}")

# Imports & Reproducibility

In [ ]:
import random
import numpy as np
import torch
import optuna
import transformers
import peft
import trl
import datasets
import bitsandbytes

from transformers import set_seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

print("--- Library Versions ---")

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Datasets:", datasets.__version__)
print("Optuna:", optuna.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

print("\nRandom seed:", SEED)

# Hugging Face Authentication

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

# Load Tokenizer

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "codellama/CodeLlama-7b-Instruct-hf"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("--- CodeLlama Tokenizer ---")
print("Model:", MODEL_ID)
print("Vocabulary size:", len(tokenizer))
print("PAD token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

# Example C# code
example = """
public string GetCustomerName(int id)
{
    return "Customer_" + id;
}
"""

tokens = tokenizer.tokenize(example)

print("\nToken count:", len(tokens))
print("Tokens:", tokens)


# Load Codellama With 4-bit NF4 Qunatization

In [ ]:
import torch
from transformers import (AutoModelForCausalLM,BitsAndBytesConfig)


## Quantization Configuration

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Loads supported weights as 4-bit.
    bnb_4bit_quant_type="nf4", # NormalFloat4 quantization
    bnb_4bit_use_double_quant=True, # compresses the quantization constants.
    bnb_4bit_compute_dtype=torch.bfloat16 # Uses BF16 for matrix calculations
)

print("Quantization configuration created.")

## Load CodeLlama Model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto", # Manages the deployment of the model onto devices
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True # Reduces CPU memory usage during model loading
)

print("\nModel loaded successfully.")

## Verify Quantization

In [ ]:
print("\n--- Model Information ---")
print("Model:", MODEL_ID)
print("Loaded in 4-bit:",model.is_loaded_in_4bit)
print("Model Device:",next(model.parameters()).device)

## GPU Memory Usage

In [ ]:
allocated_memory = (torch.cuda.memory_allocated(0) / 1024**3)
reserved_memory = (torch.cuda.memory_reserved(0) / 1024**3)
total_memory = (torch.cuda.get_device_properties(0).total_memory/ 1024**3)

print("\n--- GPU Memory ---")

print(f"Allocated: {allocated_memory:.2f} GB")
print(f"Reserved: {reserved_memory:.2f} GB")
print(f"Total: {total_memory:.2f} GB")

# Prepare Model for QLora Training

In [ ]:
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training
)

# Disable KV cache during training

In [ ]:
model.config.use_cache = False

## Prepare quantized model for training

In [ ]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True # Saves memory by recalculating activations
)

## Define initial LoRA configuration

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, # Rank value of adapter matrices
    lora_alpha=32, # The scale of the LoRA update
    lora_dropout=0.10, # Dropout rate applied to LoRA inputs
    target_modules="all-linear", # Applies LoRA to linear layers
    bias="none" # No training the bias parameters.
)

## Attach LoRA adapters

In [ ]:
model = get_peft_model(
    model,
    lora_config
)

## Inspect trainable parameters

In [ ]:
model.print_trainable_parameters()

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_percentage = (100 * trainable_params / total_params)

print("\n--- QLoRA Parameter Statistics ---")

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {trainable_percentage:.4f}%")

## GPU memory

In [ ]:
allocated = torch.cuda.memory_allocated(0) / 1024**3
print(f"\nGPU allocated: {allocated:.2f} GB")

# Create Initial Migration Dataset

In [ ]:
from datasets import Dataset

migration_examples = [
    {
        "migration_id": "config_001",
        "group_id": "configuration",
        "instruction": (
            "Migrate .NET Framework ConfigurationManager "
            "usage to modern .NET IConfiguration "
            "using constructor injection."
        ),
        "legacy_code": """
using System.Configuration;

public class AppSettings
{
    public string GetApiUrl()
    {
        return ConfigurationManager.AppSettings["ApiUrl"];
    }
}
""",
        "modern_code": """
using Microsoft.Extensions.Configuration;

public class AppSettings
{
    private readonly IConfiguration _configuration;

    public AppSettings(IConfiguration configuration)
    {
        _configuration = configuration;
    }

    public string? GetApiUrl()
    {
        return _configuration["ApiUrl"];
    }
}
"""
    },
    {
        "migration_id": "http_001",
        "group_id": "http_context",
        "instruction": (
            "Migrate ASP.NET Framework HttpContext.Current "
            "usage to ASP.NET Core by explicitly passing "
            "the request HttpContext."
        ),
        "legacy_code": """
using System.Web;

public class RequestInfo
{
    public string GetUserAgent()
    {
        return HttpContext.Current.Request.UserAgent;
    }
}
""",
        "modern_code": """
using Microsoft.AspNetCore.Http;

public class RequestInfo
{
    public string? GetUserAgent(HttpContext context)
    {
        return context.Request.Headers.UserAgent.ToString();
    }
}
"""
    }
]

# Convert to Hugging Face Dataset
dataset = Dataset.from_list(migration_examples)

print("--- Migration Dataset ---")
print("Number of examples:", len(dataset))
print("Columns:", dataset.column_names)

print("\nFirst migration example:")
print("Instruction:", dataset[0]["instruction"])
print("Legacy Code:", dataset[0]["legacy_code"])
print("Modern Code:", dataset[0]["modern_code"])

# Prepare Supervised Fine-Tuning Dataset

In [ ]:
SYSTEM_PROMPT = """
You are CodeMigrate AI, an expert C# migration assistant.

Your task is to migrate legacy .NET Framework code
to modern .NET and ASP.NET Core.

Requirements:
- Follow the migration instruction.
- Preserve the original business logic.
- Use modern .NET APIs and recommended patterns.
- Return only the migrated C# code.
- Do not include explanations or Markdown.
""".strip()


def prepare_sft_example(example):

    user_prompt = f"""
Migration instruction:
{example["instruction"]}

Legacy C# code:
{example["legacy_code"]}
""".strip()

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    completion = (
        example["modern_code"].strip()
        + tokenizer.eos_token
    )

    return {
        "prompt": prompt,
        "completion": completion
    }


sft_dataset = dataset.map(
    prepare_sft_example,
    remove_columns=dataset.column_names
)


print("--- SFT Dataset ---")
print("Examples:", len(sft_dataset))
print("Columns:", sft_dataset.column_names)

print("\n--- First Training Prompt ---")
print(sft_dataset[0]["prompt"])

print("\n--- Expected Model Completion ---")
print(sft_dataset[0]["completion"])

print("\n--- Token Counts ---")

for i, example in enumerate(sft_dataset):

    prompt_tokens = tokenizer(
        example["prompt"],
        add_special_tokens=False
    )["input_ids"]

    completion_tokens = tokenizer(
        example["completion"],
        add_special_tokens=False
    )["input_ids"]

    print(
        f"Example {i + 1}: "
        f"Prompt={len(prompt_tokens)} tokens | "
        f"Completion={len(completion_tokens)} tokens"
    )

# Collect Real .NET Migration Sources

In [ ]:
from pathlib import Path
from collections import defaultdict
import subprocess

REPO_URL = ("https://github.com/microsoft/"
    "dotnet-modernization-for-beginners.git"
)

DATA_ROOT = Path("/content/codemigrate_sources")

REPO_PATH = (DATA_ROOT / "dotnet-modernization-for-beginners")

DATA_ROOT.mkdir(parents=True, exist_ok=True)


# Download the repository once
if not REPO_PATH.exists():

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            REPO_URL,
            str(REPO_PATH)
        ],
        check=True
    )


# Source directories
LEGACY_DIR = REPO_PATH / "shared-legacy-app"

MODERN_DIR = (REPO_PATH / "examples" / "modernized" / "src")


def collect_csharp_files(directory):
    return sorted(
        file
        for file in directory.rglob("*.cs")
        if "obj" not in file.parts
        and "bin" not in file.parts
    )


legacy_files = collect_csharp_files(LEGACY_DIR)

modern_files = collect_csharp_files(MODERN_DIR)


print("--- Source Collection ---")

print("Legacy C# files:", len(legacy_files))
print("Modern C# files:", len(modern_files))


# Group files by filename
legacy_index = defaultdict(list)
modern_index = defaultdict(list)

for file in legacy_files:
    legacy_index[file.stem].append(file)

for file in modern_files:
    modern_index[file.stem].append(file)


# Candidate matches
common_names = sorted(
    set(legacy_index) & set(modern_index)
)


print("\n--- Potential Migration Pairs ---")
print("Matching filenames:", len(common_names))


for name in common_names[:15]:

    print(f"\n{name}")

    for file in legacy_index[name]:
        print(
            "  Legacy:",
            file.relative_to(REPO_PATH)
        )

    for file in modern_index[name]:
        print(
            "  Modern:",
            file.relative_to(REPO_PATH)
        )

# Build and Inspect Migration Candidates

In [ ]:
from datasets import Dataset
from difflib import unified_diff

MIGRATION_INSTRUCTIONS = {
    "ApplicationDbContext": (
        "Migrate Entity Framework 6 DbContext to "
        "Entity Framework Core using modern DbContext "
        "configuration and dependency injection."
    ),

    "Book": (
        "Migrate the legacy .NET Framework Book model "
        "to a model compatible with modern .NET."
    ),

    "BooksController": (
        "Migrate ASP.NET MVC controller code to "
        "ASP.NET Core MVC while preserving "
        "the original business functionality."
    )
}


migration_candidates = []

for name, instruction in MIGRATION_INSTRUCTIONS.items():

    legacy_paths = legacy_index[name]
    modern_paths = modern_index[name]

    # We expect one source file on each side.
    assert len(legacy_paths) == 1
    assert len(modern_paths) == 1

    legacy_file = legacy_paths[0]
    modern_file = modern_paths[0]

    legacy_code = legacy_file.read_text(
        encoding="utf-8-sig"
    )

    modern_code = modern_file.read_text(
        encoding="utf-8-sig"
    )

    migration_candidates.append({
        "migration_id": f"bookcatalog_{name}",
        "group_id": "microsoft_bookcatalog",
        "instruction": instruction,
        "legacy_code": legacy_code,
        "modern_code": modern_code,
        "review_status": "pending"
    })


candidate_dataset = Dataset.from_list(
    migration_candidates
)


print("--- Migration Candidates ---")
print("Total candidates:", len(candidate_dataset))


for example in candidate_dataset:

    legacy_code = example["legacy_code"]
    modern_code = example["modern_code"]

    legacy_tokens = len(
        tokenizer.encode(
            legacy_code,
            add_special_tokens=False
        )
    )

    modern_tokens = len(
        tokenizer.encode(
            modern_code,
            add_special_tokens=False
        )
    )

    print("\n" + "=" * 60)
    print("Migration:", example["migration_id"])
    print("Legacy tokens:", legacy_tokens)
    print("Modern tokens:", modern_tokens)

    diff = list(
        unified_diff(
            legacy_code.splitlines(keepends=True),
            modern_code.splitlines(keepends=True),
            fromfile="legacy",
            tofile="modern",
            n=2
        )
    )

    print("\n--- Code Changes (Preview) ---")
    print("".join(diff[:35]))

    if len(diff) > 35:
        print(f"... {len(diff) - 35} additional diff lines")

**Review Book Model Migration**

In [ ]:
from difflib import unified_diff

book_example = next(
    item for item in migration_candidates
    if item["migration_id"] == "bookcatalog_Book"
)

legacy_book = book_example["legacy_code"]
modern_book = book_example["modern_code"]

print("--- COMPLETE BOOK MODEL DIFF ---\n")

diff = unified_diff(
    legacy_book.splitlines(keepends=True),
    modern_book.splitlines(keepends=True),
    fromfile="Book.cs (.NET Framework)",
    tofile="Book.cs (Modern .NET)",
    n=3
)

print("".join(diff))

print("\n--- BOOK MODEL TOKEN LENGTH ---")

legacy_tokens = len(
    tokenizer.encode(
        legacy_book,
        add_special_tokens=False
    )
)

modern_tokens = len(
    tokenizer.encode(
        modern_book,
        add_special_tokens=False
    )
)

print("Legacy:", legacy_tokens)
print("Modern:", modern_tokens)
print("Total code tokens:", legacy_tokens + modern_tokens)

**Add Reviewed Migration Example**

In [ ]:
from datasets import Dataset

book_record = {
    "migration_id": "bookcatalog_Book",
    "group_id": "microsoft_bookcatalog",

    "instruction": (
        "Modernize this .NET Framework C# entity using "
        "file-scoped namespaces, nullable reference type "
        "annotations, and property initializers. "
        "Preserve data annotations and the initialization "
        "behavior of IsActive and CreatedDate."
    ),

    "legacy_code": legacy_book,
    "modern_code": modern_book
}


# Keep the original two examples and add Book.cs
migration_pool = Dataset.from_list(
    [
        *dataset.to_list(),
        book_record
    ]
)


print("--- Migration Data Pool ---")
print("Total examples:", len(migration_pool))

print("\n--- Migration Records ---")

for example in migration_pool:

    print(
        example["migration_id"],
        "| Group:",
        example["group_id"]
    )

**Controlled Data Augmentation**

In [ ]:
from collections import Counter

config_seed = next(
    row for row in migration_pool
    if row["migration_id"] == "config_001"
)

class_names = [
    "PaymentSettings",
    "EmailSettings",
    "StorageSettings",
    "SecuritySettings"
]

settings = [
    ("GetPaymentEndpoint", "PaymentEndpoint"),
    ("GetSmtpHost", "SmtpHost"),
    ("GetStoragePath", "StoragePath"),
    ("GetAuthServer", "AuthServer"),
    ("GetClientIdentifier", "ClientIdentifier"),
    ("GetServiceEndpoint", "ServiceEndpoint")
]


def rename_config_identifiers(
    code,
    class_name,
    method_name,
    config_key
):

    # Rename only our custom class
    code = code.replace(
        "class AppSettings",
        f"class {class_name}"
    )

    # Rename its constructor
    code = code.replace(
        "public AppSettings(",
        f"public {class_name}("
    )

    # Rename the method
    code = code.replace(
        "GetApiUrl",
        method_name
    )

    # Rename the configuration key
    code = code.replace(
        '"ApiUrl"',
        f'"{config_key}"'
    )

    return code


augmented_examples = []

for class_name in class_names:

    for method_name, config_key in settings:

        legacy_code = rename_config_identifiers(
            config_seed["legacy_code"],
            class_name,
            method_name,
            config_key
        )

        modern_code = rename_config_identifiers(
            config_seed["modern_code"],
            class_name,
            method_name,
            config_key
        )

        # Validate the key migration pattern
        assert (
            f'ConfigurationManager.AppSettings["{config_key}"]'
            in legacy_code
        ), "Legacy ConfigurationManager API was corrupted!"

        assert (
            f'_configuration["{config_key}"]'
            in modern_code
        ), "Modern IConfiguration API was corrupted!"

        assert f"class {class_name}" in legacy_code
        assert f"class {class_name}" in modern_code
        assert f"public {class_name}(" in modern_code

        augmented_examples.append({
            "migration_id": (
                f"config_aug_{len(augmented_examples) + 1:03d}"
            ),
            "group_id": config_seed["group_id"],
            "instruction": config_seed["instruction"],
            "legacy_code": legacy_code,
            "modern_code": modern_code
        })


assert len({
    row["legacy_code"]
    for row in augmented_examples
}) == len(augmented_examples)


# Rebuild the training pool
training_pool = Dataset.from_list(
    [
        *migration_pool.to_list(),
        *augmented_examples
    ]
)


print("--- Corrected Data Augmentation ---")
print("Original examples:", len(migration_pool))
print("Augmented examples:", len(augmented_examples))
print("Total examples:", len(training_pool))

print("\n--- Group Distribution ---")
print(Counter(training_pool["group_id"]))

print("\n--- Corrected Legacy Code ---")
print(augmented_examples[0]["legacy_code"])

print("\n--- Corrected Modern Code ---")
print(augmented_examples[0]["modern_code"])

print("\nAll API preservation checks: PASS")

# Collect a Second Migration Source

In [ ]:
from pathlib import Path
import subprocess

SECOND_REPO_URL = (
    "https://github.com/theekavi/"
    "dotnet-modernization.git"
)

SECOND_REPO = (DATA_ROOT / "dotnet-modernization-case-study")


if not SECOND_REPO.exists():

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            SECOND_REPO_URL,
            str(SECOND_REPO)
        ],
        check=True
    )


SECOND_LEGACY_DIR = SECOND_REPO / "legacy"

SECOND_MODERN_DIR = (SECOND_REPO / "modern" / "src" / "OrderApi")


second_legacy_files = collect_csharp_files(SECOND_LEGACY_DIR)

second_modern_files = collect_csharp_files(SECOND_MODERN_DIR)


print("--- Second Migration Source ---")

print("Legacy C# files:",len(second_legacy_files))

print("Modern C# files:",len(second_modern_files))


print("\n--- Legacy Files ---")

for file in second_legacy_files:

    print(file.relative_to(SECOND_LEGACY_DIR))


print("\n--- Modern Files ---")

for file in second_modern_files:

    print(file.relative_to(SECOND_MODERN_DIR))

**Inspect Migration Source Quality**

In [ ]:
files_to_inspect = [
    ("LEGACY", SECOND_LEGACY_DIR / "Controllers/OrdersController.cs"),
    ("LEGACY", SECOND_LEGACY_DIR / "Models/OrderDto.cs"),
    ("LEGACY", SECOND_LEGACY_DIR / "Services/Services.cs"),

    ("MODERN", SECOND_MODERN_DIR / "Endpoints/OrderEndpoints.cs"),
    ("MODERN", SECOND_MODERN_DIR / "Models/Order.cs"),
    ("MODERN", SECOND_MODERN_DIR / "Services/OrderService.cs")
]


suspicious_patterns = [
    "TODO",
    "NotImplementedException",
    "throw new NotImplementedException",
    "// ...",
    "/* ... */"
]


for source_type, file_path in files_to_inspect:

    code = file_path.read_text(
        encoding="utf-8-sig"
    )

    lines = code.splitlines()

    token_count = len(
        tokenizer.encode(
            code,
            add_special_tokens=False
        )
    )

    flags = [
        pattern
        for pattern in suspicious_patterns
        if pattern in code
    ]

    print("\n" + "=" * 65)
    print("Source:", source_type)
    print("File:", file_path.name)
    print("Lines:", len(lines))
    print("Tokens:", token_count)
    print("Quality flags:", flags or "None")

    print("\n--- Code Preview ---")

    for number, line in enumerate(lines[:40], start=1):
        print(f"{number:3}: {line}")

    if len(lines) > 40:
        print(f"... {len(lines) - 40} more lines")

# Extract Method-Level Migration Candidates

In [ ]:
import re

def extract_csharp_method(source_code, method_name):

    pattern = re.compile(
        rf"(?m)^[ \t]*public\s+"
        rf"(?:async\s+)?"
        rf"[\w<>,?\[\]]+\s+"
        rf"{re.escape(method_name)}\s*"
        rf"\([^)]*\)\s*\{{"
    )

    match = pattern.search(source_code)

    if match is None:
        raise ValueError(
            f"Method not found: {method_name}"
        )

    opening_index = source_code.find(
        "{",
        match.start(),
        match.end()
    )

    depth = 0

    for i in range(opening_index, len(source_code)):

        character = source_code[i]

        if character == "{":
            depth += 1

        elif character == "}":
            depth -= 1

            if depth == 0:
                return source_code[
                    match.start():i + 1
                ].strip()

    raise ValueError(
        f"Unclosed method body: {method_name}"
    )


# Load Microsoft BookCatalog controllers
legacy_controller = legacy_index[
    "BooksController"
][0].read_text(encoding="utf-8-sig")

modern_controller = modern_index[
    "BooksController"
][0].read_text(encoding="utf-8-sig")


method_candidates = []

for method_name in ["Index", "Details"]:

    legacy_method = extract_csharp_method(
        legacy_controller,
        method_name
    )

    modern_method = extract_csharp_method(
        modern_controller,
        method_name
    )

    method_candidates.append({
        "method_name": method_name,
        "legacy_code": legacy_method,
        "modern_code": modern_method
    })


print("--- Method-Level Migration Candidates ---")
print("Total methods:", len(method_candidates))


for candidate in method_candidates:

    print("\n" + "=" * 60)
    print("Method:", candidate["method_name"])

    print("\n--- Legacy Method ---")
    print(candidate["legacy_code"])

    print("\n--- Modern Method ---")
    print(candidate["modern_code"])

**Add Method-Level Migration Examples**

In [ ]:
METHOD_INSTRUCTIONS = {

    "Index": (
        "Migrate this ASP.NET MVC Index action to "
        "ASP.NET Core MVC and Entity Framework Core. "
        "The modern controller receives its DbContext "
        "through constructor injection as 'db'. "
        "Use asynchronous EF Core queries and "
        "AsNoTracking for read-only access. "
        "Preserve the active-book filter, title ordering, "
        "ViewBag.UserAgent, and View result. "
        "Return the migrated C# method."
    ),

    "Details": (
        "Migrate this ASP.NET MVC Details action to "
        "ASP.NET Core MVC and Entity Framework Core. "
        "The modern controller receives its DbContext "
        "through constructor injection as 'db'. "
        "Use asynchronous entity lookup. "
        "Preserve the HTTP 404 response when the book "
        "does not exist and return the View when found. "
        "Return the migrated C# method."
    )
}


method_examples = []

for candidate in method_candidates:

    method_name = candidate["method_name"]

    method_examples.append({

        "migration_id": (
            f"bookcatalog_BooksController_{method_name}"
        ),

        "group_id": "microsoft_bookcatalog",

        "instruction": METHOD_INSTRUCTIONS[method_name],

        "legacy_code": candidate["legacy_code"],

        "modern_code": candidate["modern_code"]
    })


# Rebuild from the original sources to avoid duplicates
training_pool = Dataset.from_list(
    [
        *migration_pool.to_list(),
        *augmented_examples,
        *method_examples
    ]
)


# Check migration IDs
migration_ids = training_pool["migration_id"]

assert len(migration_ids) == len(set(migration_ids))


print("--- Updated Training Pool ---")

print("Total examples:", len(training_pool))

print("\n--- Group Distribution ---")
print(Counter(training_pool["group_id"]))


print("\n--- Added Method Examples ---")

for example in method_examples:

    legacy_tokens = len(
        tokenizer.encode(
            example["legacy_code"],
            add_special_tokens=False
        )
    )

    modern_tokens = len(
        tokenizer.encode(
            example["modern_code"],
            add_special_tokens=False
        )
    )

    print(
        f'{example["migration_id"]} | '
        f'Legacy: {legacy_tokens} | '
        f'Modern: {modern_tokens}'
    )


print("\nMethod-level examples added: PASS")

29 examples have been successfully added. We will now conclude the data preparation phase and move on to the QLoRA training pipeline.

First, we will conduct a brief pilot training run. Our goal is to verify the GPU memory usage, data format, loss calculation, and whether the LoRA weights are actually being updated. Following this, we will carry out the Optuna experiments, starting from a fresh, clean set of initial weights.

# Configure QLoRA Pilot Training

In [ ]:
from trl import SFTConfig, SFTTrainer

# Convert all 29 examples to prompt-completion format
pilot_sft_dataset = training_pool.map(
    prepare_sft_example,
    remove_columns=training_pool.column_names
)


# Measure full sequence lengths
sequence_lengths = []

for example in pilot_sft_dataset:

    token_ids = tokenizer.encode(
        example["prompt"] + example["completion"],
        add_special_tokens=False
    )

    sequence_lengths.append(len(token_ids))


print("--- Sequence Length Statistics ---")
print("Examples:", len(sequence_lengths))
print("Minimum:", min(sequence_lengths))
print("Median:", int(np.median(sequence_lengths)))
print("Maximum:", max(sequence_lengths))


# Ensure no training example is truncated
assert max(sequence_lengths) <= 1024, (
    "Some examples exceed 1024 tokens. "
    "Review sequence lengths before training."
)

MAX_LENGTH = max(
    512,
    ((max(sequence_lengths) + 63) // 64) * 64
)


# CodeLlama already has its LoRA adapter attached
model.config.use_cache = False
tokenizer.padding_side = "right"


pilot_config = SFTConfig(
    output_dir="/content/codemigrate_qlora_pilot",

    max_length=MAX_LENGTH,

    # GPU memory management
    per_device_train_batch_size=1, # We use the L4 GPU's memory in a controlled manner.
    gradient_accumulation_steps=4, # We accumulate gradients over four mini-batches. On a single GPU, the effective batch size is approximately 4.
    bf16=True, # L4 tarafından desteklenen BF16 hesaplamalarını kullanıyoruz.
    gradient_checkpointing=True,

    # Small pilot run
    max_steps=12, # This is just a short pilot. It is not the final training duration.

    # Initial values; Optuna will tune these later
    learning_rate=2e-4, # Initial trial value. Optuna will subsequently compare the parameters we have specified, including the learning rate.
    weight_decay=0.01,
    warmup_steps=2,
    lr_scheduler_type="cosine",

    optim="paged_adamw_8bit",

    # Train only on migrated code tokens
    completion_only_loss=True,
    packing=False,

    logging_steps=1,
    save_strategy="no",
    report_to="none",

    dataloader_num_workers=0
)


pilot_trainer = SFTTrainer(
    model=model,
    args=pilot_config,
    train_dataset=pilot_sft_dataset,
    processing_class=tokenizer
)


# Inspect the batch before training
batch = next(
    iter(pilot_trainer.get_train_dataloader())
)

labels = batch["labels"]

active_tokens = int(
    (labels != -100).sum().item()
)

masked_tokens = int(
    (labels == -100).sum().item()
)


print("\n--- Training Configuration ---")
print("Maximum sequence length:", MAX_LENGTH)
print("Batch size:", pilot_config.per_device_train_batch_size)
print("Gradient accumulation:", pilot_config.gradient_accumulation_steps)
print("Maximum steps:", pilot_config.max_steps)
print("Learning rate:", pilot_config.learning_rate)

print("\n--- Loss Mask Inspection ---")
print("Tokens contributing to loss:", active_tokens)
print("Masked tokens:", masked_tokens)

assert active_tokens > 0
assert masked_tokens > 0

print("\nTraining batch preparation: PASS")

## Run QLoRA Pilot Training

In [ ]:
import time
import torch


# Track one trainable LoRA matrix
tracked_name, tracked_parameter = next(
    (name, parameter)
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
    and "lora_B" in name
)


# Save its initial values
weights_before = (
    tracked_parameter
    .detach()
    .float()
    .cpu()
    .clone()
)


# Reset GPU peak memory statistics
torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()


# Start training
pilot_result = pilot_trainer.train()


training_seconds = (time.perf_counter() - start_time)


# Compare weights after training
weights_after = (
    tracked_parameter
    .detach()
    .float()
    .cpu()
    .clone()
)


weight_difference = (weights_after - weights_before).abs()


peak_memory_gb = (torch.cuda.max_memory_allocated()/ 1024**3)


print("\n--- QLoRA Pilot Results ---")

print("Training loss:",pilot_result.metrics.get("train_loss"))

print("Completed optimizer steps:",pilot_trainer.state.global_step)

print("Training time:",round(training_seconds / 60, 2),"minutes")

print("Peak GPU memory:",round(peak_memory_gb, 2),"GB")


print("\n--- LoRA Weight Update ---")

print("Tracked parameter:", tracked_name)

print("Mean absolute weight change:",weight_difference.mean().item())

print("Maximum absolute weight change:",weight_difference.max().item())


assert pilot_trainer.state.global_step == 12

assert weight_difference.max().item() > 0, ("LoRA weights were not updated!")


print("\nQLoRA pilot training: PASS")

# Save Pilot Adapter and Restore Base Model

In [ ]:
import gc

# 1. Save trained LoRA adapter
PILOT_ADAPTER_PATH = Path("/content/codemigrate_qlora_pilot_adapter")

model.save_pretrained(str(PILOT_ADAPTER_PATH),safe_serialization=True)

tokenizer.save_pretrained(str(PILOT_ADAPTER_PATH))

print("--- Pilot Adapter Saved ---")
print("Path:", PILOT_ADAPTER_PATH)


# 2. Release trainer and optimizer memory
del pilot_trainer
del tracked_parameter

gc.collect()


# 3. Remove trained LoRA layers WITHOUT merging
base_model = model.unload()

del model

gc.collect()
torch.cuda.empty_cache()


# 4. Verify restored base model
lora_parameters = [
    name
    for name, _ in base_model.named_parameters()
    if "lora_A" in name or "lora_B" in name
]

assert len(lora_parameters) == 0, ("LoRA layers still exist in the base model.")


print("\n--- Restored Base Model ---")

print("Model:", MODEL_ID)

print("Loaded in 4-bit:",base_model.is_loaded_in_4bit)

print("Remaining LoRA parameters:",len(lora_parameters))

print("Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in base_model.parameters()
        if parameter.requires_grad
    )
)

print("GPU allocated:",round(torch.cuda.memory_allocated() / 1024**3,2),"GB")

print("\nBase model restoration: PASS")

# Creating Validation Candidates

In [ ]:
validation_examples = [

    # -------------------------------------------------
    # 1. ConfigurationManager: typed value with fallback
    # -------------------------------------------------
    {
        "migration_id": "val_config_retry",
        "group_id": "val_config_numeric",

        "instruction": (
            "Migrate ConfigurationManager.AppSettings to "
            "IConfiguration using constructor injection. "
            "Preserve integer parsing and the default value."
        ),

        "legacy_code": """
using System.Configuration;

public class RetryPolicy
{
    public int GetRetryCount()
    {
        var raw = ConfigurationManager.AppSettings["RetryCount"];

        return int.TryParse(raw, out int count)
            ? count
            : 3;
    }
}
""",

        "modern_code": """
using Microsoft.Extensions.Configuration;

public class RetryPolicy
{
    private readonly IConfiguration _configuration;

    public RetryPolicy(IConfiguration configuration)
    {
        _configuration = configuration;
    }

    public int GetRetryCount()
    {
        var raw = _configuration["RetryCount"];

        return int.TryParse(raw, out int count)
            ? count
            : 3;
    }
}
"""
    },

    # -------------------------------------------------
    # 2. HttpContext: query string migration
    # -------------------------------------------------
    {
        "migration_id": "val_http_query",
        "group_id": "val_http_query",

        "instruction": (
            "Migrate System.Web.HttpContext query string access "
            "to ASP.NET Core HttpContext. "
            "Accept HttpContext as a method parameter. "
            "Return null when the query parameter is missing."
        ),

        "legacy_code": """
using System.Web;

public class TrackingReader
{
    public string GetTrackingId()
    {
        return HttpContext.Current.Request.QueryString["tracking"];
    }
}
""",

        "modern_code": """
using Microsoft.AspNetCore.Http;

public class TrackingReader
{
    public string? GetTrackingId(HttpContext context)
    {
        return context.Request.Query.TryGetValue(
            "tracking", out var values
        )
            ? values.ToString()
            : null;
    }
}
"""
    },

    # -------------------------------------------------
    # 3. HttpContext: cookie migration
    # -------------------------------------------------
    {
        "migration_id": "val_http_cookie",
        "group_id": "val_http_cookie",

        "instruction": (
            "Migrate System.Web.HttpContext cookie access "
            "to ASP.NET Core HttpContext. "
            "Accept HttpContext as a method parameter. "
            "Preserve null when the cookie is missing."
        ),

        "legacy_code": """
using System.Web;

public class PreferenceReader
{
    public string GetTheme()
    {
        return HttpContext.Current.Request.Cookies["theme"]?.Value;
    }
}
""",

        "modern_code": """
using Microsoft.AspNetCore.Http;

public class PreferenceReader
{
    public string? GetTheme(HttpContext context)
    {
        return context.Request.Cookies.TryGetValue(
            "theme", out var value
        )
            ? value
            : null;
    }
}
"""
    },

    # -------------------------------------------------
    # 4. Entity Framework: asynchronous query
    # -------------------------------------------------
    {
        "migration_id": "val_ef_count",
        "group_id": "val_ef_count",

        "instruction": (
            "Migrate this Entity Framework 6 method to "
            "Entity Framework Core using asynchronous database "
            "access. The modern class already receives the "
            "DbContext through constructor injection as 'db'. "
            "Preserve the active-order filter. "
            "Return only the migrated C# method."
        ),

        "legacy_code": """
public int CountActiveOrders()
{
    return db.Orders.Count(o => o.IsActive);
}
""",

        "modern_code": """
public async Task<int> CountActiveOrders()
{
    return await db.Orders.CountAsync(o => o.IsActive);
}
"""
    }
]


validation_candidates = Dataset.from_list(
    validation_examples
)


# Ensure no exact legacy snippet exists in training
training_code = {
    row["legacy_code"].strip()
    for row in training_pool
}

for example in validation_candidates:

    assert (
        example["legacy_code"].strip()
        not in training_code
    ), "Exact training/validation overlap detected."


print("--- Validation Candidates ---")
print("Examples:", len(validation_candidates))

for example in validation_candidates:

    print(
        example["migration_id"],
        "| Group:",
        example["group_id"]
    )

print("\nExact-overlap check: PASS")

# Optuna Objective for QLoRA

In [ ]:
import gc
import random
import numpy as np
import torch
import optuna

from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer


# Prepare datasets
optuna_train_dataset = pilot_sft_dataset

optuna_val_dataset = validation_candidates.map(
    prepare_sft_example,
    remove_columns=validation_candidates.column_names
)


# Ensure validation examples fit our context length
for example in optuna_val_dataset:

    length = len(
        tokenizer.encode(
            example["prompt"] + example["completion"],
            add_special_tokens=False
        )
    )

    assert length <= MAX_LENGTH


def objective(trial):

    global base_model

    # -----------------------------------------
    # 1. Hyperparameter search space
    # -----------------------------------------

    lora_r = trial.suggest_categorical(
        "lora_r",
        [8, 16, 32]
    )

    alpha_ratio = trial.suggest_categorical(
        "alpha_ratio",
        [1, 2, 4]
    )

    lora_alpha = lora_r * alpha_ratio

    lora_dropout = trial.suggest_float(
        "lora_dropout",
        0.0,
        0.15,
        step=0.05
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-5,
        3e-4,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        0.0,
        0.1,
        step=0.02
    )

    warmup_ratio = trial.suggest_float(
        "warmup_ratio",
        0.0,
        0.15,
        step=0.05
    )

    scheduler = trial.suggest_categorical(
        "scheduler",
        ["linear", "cosine"]
    )

    # -----------------------------------------
    # 2. Reproducibility
    # -----------------------------------------

    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)

    # -----------------------------------------
    # 3. Fresh LoRA adapter
    # -----------------------------------------

    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules="all-linear",
        bias="none",
        task_type="CAUSAL_LM"
    )

    trial_model = None
    trainer = None

    try:

        trial_model = get_peft_model(
            base_model,
            lora_config
        )

        trial_model.config.use_cache = False

        # -------------------------------------
        # 4. Training configuration
        # -------------------------------------

        training_config = SFTConfig(
            output_dir=(
                f"/content/codemigrate_optuna/"
                f"trial_{trial.number}"
            ),

            max_length=MAX_LENGTH,

            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,

            max_steps=12,

            learning_rate=learning_rate,
            weight_decay=weight_decay,
            warmup_steps=int(round(12 * warmup_ratio)),
            lr_scheduler_type=scheduler,

            optim="paged_adamw_8bit",

            bf16=True,
            gradient_checkpointing=True,

            completion_only_loss=True,
            packing=False,

            eval_strategy="no",

            logging_steps=4,
            save_strategy="no",
            report_to="none",

            seed=42,
            data_seed=42
        )

        # -------------------------------------
        # 5. Create trainer
        # -------------------------------------

        trainer = SFTTrainer(
            model=trial_model,
            args=training_config,
            train_dataset=optuna_train_dataset,
            eval_dataset=optuna_val_dataset,
            processing_class=tokenizer
        )

        # -------------------------------------
        # 6. Train
        # -------------------------------------

        train_result = trainer.train()

        # -------------------------------------
        # 7. Validate
        # -------------------------------------

        eval_result = trainer.evaluate()

        validation_loss = float(
            eval_result["eval_loss"]
        )

        trial.set_user_attr(
            "training_loss",
            float(train_result.training_loss)
        )

        trial.set_user_attr(
            "lora_alpha",
            lora_alpha
        )

        trial.set_user_attr(
            "validation_loss",
            validation_loss
        )

        return validation_loss

    finally:

        # -------------------------------------
        # 8. Remove adapter between trials
        # -------------------------------------

        if trainer is not None:
            del trainer

        if trial_model is not None:

            base_model = trial_model.unload()

            del trial_model

        gc.collect()
        torch.cuda.empty_cache()


print("--- Optuna Objective Ready ---")

print("Training examples:",len(optuna_train_dataset))

print("Validation examples:",len(optuna_val_dataset))

print("Max sequence length:",MAX_LENGTH)

print(
    "Frozen base model:",
    sum(
        parameter.numel()
        for parameter in base_model.parameters()
        if parameter.requires_grad
    ) == 0
)

print("\nOptuna objective defined successfully")

**Run Optuna Hyperparameter Search**

In [ ]:
import gc
import random
import inspect
import optuna
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
from optuna.trial import TrialState


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
MAX_STEPS = 12
TARGET_TRIALS = 8

STUDY_NAME = "codemigrate_qlora_pilot_v3"

STORAGE_URL = (
    "sqlite:////content/codemigrate_optuna_pilot.db"
)

OUTPUT_DIR = "/content/codemigrate_optuna_v3"


# ============================================================
# 2. DATASETS
# ============================================================

optuna_train_dataset = pilot_sft_dataset

optuna_val_dataset = validation_candidates.map(
    prepare_sft_example,
    remove_columns=validation_candidates.column_names
)


print("--- Dataset Information ---")

print("Training examples:", len(optuna_train_dataset))
print("Validation examples:", len(optuna_val_dataset))
print("Maximum sequence length:", MAX_LENGTH)


# Check sequence lengths
for example in optuna_val_dataset:

    token_ids = tokenizer.encode(
        example["prompt"] + example["completion"],
        add_special_tokens=False
    )

    assert len(token_ids) <= MAX_LENGTH, (
        "Validation sequence exceeds MAX_LENGTH"
    )


# ============================================================
# 3. CHECK LIBRARY COMPATIBILITY
# ============================================================

sft_parameters = set(
    inspect.signature(SFTConfig).parameters
)

required_parameters = {
    "warmup_steps",
    "max_steps",
    "learning_rate",
    "weight_decay",
    "lr_scheduler_type",
    "gradient_checkpointing",
    "completion_only_loss",
    "eval_strategy"
}

missing_parameters = (
    required_parameters - sft_parameters
)

assert not missing_parameters, (
    f"Unsupported SFTConfig parameters: {missing_parameters}"
)


print("\n--- Library Compatibility ---")
print("warmup_steps supported:", "warmup_steps" in sft_parameters)
print("SFTConfig compatibility: PASS")


# ============================================================
# 4. CHECK BASE MODEL
# ============================================================

assert not any(
    "lora_A" in name or "lora_B" in name
    for name, _ in base_model.named_parameters()
), "Base model still contains LoRA layers"


assert not any(
    parameter.requires_grad
    for parameter in base_model.parameters()
), "Base model must be frozen"


print("\n--- Base Model ---")
print("Loaded in 4-bit:", base_model.is_loaded_in_4bit)
print("Frozen base model: True")


# ============================================================
# 5. OPTUNA OBJECTIVE
# ============================================================

def objective(trial):

    global base_model

    # --------------------------------------------------------
    # Hyperparameter Search Space
    # --------------------------------------------------------

    lora_r = trial.suggest_categorical(
        "lora_r",
        [8, 16, 32]
    )

    alpha_ratio = trial.suggest_categorical(
        "alpha_ratio",
        [1, 2, 4]
    )

    lora_alpha = lora_r * alpha_ratio

    lora_dropout = trial.suggest_float(
        "lora_dropout",
        0.0,
        0.15,
        step=0.05
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-5,
        3e-4,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        0.0,
        0.1,
        step=0.02
    )

    warmup_ratio = trial.suggest_float(
        "warmup_ratio",
        0.0,
        0.15,
        step=0.05
    )

    scheduler = trial.suggest_categorical(
        "scheduler",
        ["linear", "cosine"]
    )

    # IMPORTANT:
    # Convert Optuna warmup_ratio into supported warmup_steps.

    warmup_steps = int(
        round(MAX_STEPS * warmup_ratio)
    )

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    random.seed(SEED)
    np.random.seed(SEED)

    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    # --------------------------------------------------------
    # Fresh LoRA Configuration
    # --------------------------------------------------------

    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules="all-linear",
        bias="none",
        task_type="CAUSAL_LM"
    )

    trial_model = None
    trainer = None

    try:

        # ----------------------------------------------------
        # Attach fresh LoRA adapter
        # ----------------------------------------------------

        trial_model = get_peft_model(
            base_model,
            lora_config
        )

        trial_model.config.use_cache = False

        # ----------------------------------------------------
        # Training Arguments
        # ----------------------------------------------------

        training_config = SFTConfig(

            output_dir=(
                f"{OUTPUT_DIR}/trial_{trial.number}"
            ),

            max_length=MAX_LENGTH,

            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,

            gradient_accumulation_steps=4,

            max_steps=MAX_STEPS,

            learning_rate=learning_rate,
            weight_decay=weight_decay,

            # FIXED: use warmup_steps, NOT warmup_ratio
            warmup_steps=warmup_steps,

            lr_scheduler_type=scheduler,

            optim="paged_adamw_8bit",

            bf16=True,
            gradient_checkpointing=True,

            completion_only_loss=True,
            packing=False,

            eval_strategy="no",

            logging_steps=4,
            save_strategy="no",
            report_to="none",

            dataloader_num_workers=0,

            seed=SEED,
            data_seed=SEED
        )

        # ----------------------------------------------------
        # Initialize Trainer
        # ----------------------------------------------------

        trainer = SFTTrainer(
            model=trial_model,
            args=training_config,
            train_dataset=optuna_train_dataset,
            eval_dataset=optuna_val_dataset,
            processing_class=tokenizer
        )

        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        print(
            f"\n--- Trial {trial.number} Training ---"
        )

        train_result = trainer.train()

        # ----------------------------------------------------
        # Evaluate
        # ----------------------------------------------------

        eval_result = trainer.evaluate()

        validation_loss = float(
            eval_result["eval_loss"]
        )

        training_loss = float(
            train_result.training_loss
        )

        # ----------------------------------------------------
        # Save Trial Metadata
        # ----------------------------------------------------

        trial.set_user_attr(
            "training_loss",
            training_loss
        )

        trial.set_user_attr(
            "validation_loss",
            validation_loss
        )

        trial.set_user_attr(
            "lora_alpha",
            lora_alpha
        )

        trial.set_user_attr(
            "warmup_steps",
            warmup_steps
        )

        print(
            f"\nTrial {trial.number} completed"
        )

        print(
            f"Training Loss: {training_loss:.6f}"
        )

        print(
            f"Validation Loss: {validation_loss:.6f}"
        )

        return validation_loss

    finally:

        # ----------------------------------------------------
        # Clean Up LoRA Between Trials
        # ----------------------------------------------------

        if trainer is not None:
            del trainer

        if trial_model is not None:

            # Remove LoRA without merging into base model
            base_model = trial_model.unload()

            del trial_model

        gc.collect()

        torch.cuda.empty_cache()


# ============================================================
# 6. OPTUNA SAMPLER
# ============================================================

sampler = optuna.samplers.TPESampler(
    seed=SEED,
    n_startup_trials=3
)


# ============================================================
# 7. CREATE STUDY
# ============================================================

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_URL,
    direction="minimize",
    sampler=sampler,
    load_if_exists=True
)


# ============================================================
# 8. BASELINE TRIAL
# ============================================================

baseline_params = {
    "lora_r": 16,
    "alpha_ratio": 2,
    "lora_dropout": 0.10,
    "learning_rate": 2e-4,
    "weight_decay": 0.02,
    "warmup_ratio": 0.10,
    "scheduler": "cosine"
}

study.enqueue_trial(
    baseline_params,
    skip_if_exists=True
)


# ============================================================
# 9. RUN OPTIMIZATION
# ============================================================

completed_before = sum(
    trial.state == TrialState.COMPLETE
    for trial in study.trials
)

remaining_trials = max(
    0,
    TARGET_TRIALS - completed_before
)


print("\n--- Starting Optuna Optimization ---")

print("Study:", STUDY_NAME)
print("Target trials:", TARGET_TRIALS)
print("Completed trials:", completed_before)
print("Remaining trials:", remaining_trials)


if remaining_trials > 0:

    study.optimize(
        objective,
        n_trials=remaining_trials,
        gc_after_trial=True,
        show_progress_bar=True
    )


# ============================================================
# 10. COLLECT RESULTS
# ============================================================

completed_trials = sorted(
    [
        trial
        for trial in study.trials
        if trial.state == TrialState.COMPLETE
    ],
    key=lambda trial: trial.value
)


print("\n--- Optuna Trial Results ---")

for trial in completed_trials:

    print(
        f"Trial {trial.number:02d} | "
        f"Validation Loss: {trial.value:.6f} | "
        f"Training Loss: "
        f"{trial.user_attrs.get('training_loss', float('nan')):.6f}"
    )


# ============================================================
# 11. BEST HYPERPARAMETERS
# ============================================================

if completed_trials:

    best_trial = study.best_trial

    print("\n--- Best Trial ---")

    print(
        "Trial number:",
        best_trial.number
    )

    print(
        "Best validation loss:",
        best_trial.value
    )

    print("\n--- Best Hyperparameters ---")

    for name, value in best_trial.params.items():
        print(f"{name}: {value}")

    best_lora_alpha = (
        best_trial.params["lora_r"]
        * best_trial.params["alpha_ratio"]
    )

    print(
        "Derived LoRA alpha:",
        best_lora_alpha
    )

else:

    print("\nNo completed trials available.")


# ============================================================
# 12. FINAL MODEL VERIFICATION
# ============================================================

remaining_lora_layers = sum(
    "lora_A" in name or "lora_B" in name
    for name, _ in base_model.named_parameters()
)

assert remaining_lora_layers == 0

print("\nBase model remained clean: PASS")
print("Optuna optimization finished.")

#  Evaluate Original CodeLlama Before Fine-Tuning

In [ ]:

import gc
import torch

from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer


# --------------------------------------------------
# 1. Verify frozen base model
# --------------------------------------------------

assert not any(
    "lora_A" in name or "lora_B" in name
    for name, _ in base_model.named_parameters()
)

print("--- Base Model Evaluation ---")
print("Model:", MODEL_ID)
print("Loaded in 4-bit:", base_model.is_loaded_in_4bit)


# --------------------------------------------------
# 2. Create an untrained identity LoRA adapter
# --------------------------------------------------

identity_lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM"
)

identity_model = None
baseline_trainer = None

try:

    identity_model = get_peft_model(
        base_model,
        identity_lora_config
    )

    identity_model.config.use_cache = False

    # --------------------------------------------------
    # 3. Verify LoRA B starts at zero
    # --------------------------------------------------

    lora_b_parameters = [
        parameter
        for name, parameter in identity_model.named_parameters()
        if "lora_B" in name
    ]

    assert len(lora_b_parameters) > 0

    assert all(
        torch.count_nonzero(parameter).item() == 0
        for parameter in lora_b_parameters
    )

    print("\nUntrained LoRA B matrices: zero")
    print("Identity adapter verification: PASS")

    # --------------------------------------------------
    # 4. Evaluation configuration
    # --------------------------------------------------

    baseline_config = SFTConfig(
        output_dir="/content/codemigrate_base_evaluation",

        max_length=MAX_LENGTH,

        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,

        bf16=True,

        completion_only_loss=True,
        packing=False,

        eval_strategy="no",
        save_strategy="no",
        report_to="none",

        dataloader_num_workers=0,

        seed=42,
        data_seed=42
    )

    # --------------------------------------------------
    # 5. Initialize evaluator
    # --------------------------------------------------

    baseline_trainer = SFTTrainer(
        model=identity_model,
        args=baseline_config,
        train_dataset=optuna_train_dataset,
        eval_dataset=optuna_val_dataset,
        processing_class=tokenizer
    )

    # --------------------------------------------------
    # 6. Evaluate WITHOUT training
    # --------------------------------------------------

    baseline_metrics = baseline_trainer.evaluate()

    base_validation_loss = float(
        baseline_metrics["eval_loss"]
    )

    # --------------------------------------------------
    # 7. Compare with Optuna
    # --------------------------------------------------

    tuned_validation_loss = float(
        study.best_value
    )

    difference = (
        base_validation_loss - tuned_validation_loss
    )

    relative_difference = (
        difference / base_validation_loss * 100
    )

    print("\n--- Base vs QLoRA ---")

    print(
        "Original CodeLlama validation loss:",
        round(base_validation_loss, 6)
    )

    print(
        "Optuna QLoRA validation loss:",
        round(tuned_validation_loss, 6)
    )

    print(
        "Relative loss reduction (%):",
        round(relative_difference, 2)
    )

    if difference > 0:
        print(
            "\nQLoRA achieved lower validation loss."
        )
    else:
        print(
            "\nOriginal CodeLlama achieved equal or lower "
            "validation loss."
        )

finally:

    # --------------------------------------------------
    # 8. Remove identity adapter
    # --------------------------------------------------

    if baseline_trainer is not None:
        del baseline_trainer

    if identity_model is not None:

        base_model = identity_model.unload()

        del identity_model

    gc.collect()
    torch.cuda.empty_cache()


# --------------------------------------------------
# 9. Verify model remains clean
# --------------------------------------------------

assert not any(
    "lora_A" in name or "lora_B" in name
    for name, _ in base_model.named_parameters()
)

print("\nBase model restoration: PASS")

# Train and Save Two QLoRA Candidate Models

In [ ]:
# --------------------------------------------------
# 1. Candidate Configurations
# --------------------------------------------------

best_params = study.best_trial.params.copy()

mc_params = best_params.copy()
mc_params["lora_dropout"] = 0.10


CANDIDATES = {
    "optuna_best": best_params,
    "mc_dropout": mc_params
}


ADAPTER_ROOT = Path(
    "/content/codemigrate_adapters"
)

ADAPTER_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------
# 2. Training Function
# --------------------------------------------------

def train_and_save_candidate(candidate_name, params):

    global base_model

    SEED = 42
    TRAIN_STEPS = 12

    random.seed(SEED)
    np.random.seed(SEED)

    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    lora_r = params["lora_r"]

    lora_alpha = (
        params["lora_r"]
        * params["alpha_ratio"]
    )

    dropout = params["lora_dropout"]

    warmup_steps = int(
        round(
            TRAIN_STEPS * params["warmup_ratio"]
        )
    )

    # ----------------------------------------------
    # Create fresh LoRA configuration
    # ----------------------------------------------

    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=dropout,
        target_modules="all-linear",
        bias="none",
        task_type="CAUSAL_LM"
    )

    candidate_model = None
    trainer = None

    start_time = time.perf_counter()

    try:

        candidate_model = get_peft_model(
            base_model,
            lora_config
        )

        candidate_model.config.use_cache = False

        # ------------------------------------------
        # Training configuration
        # ------------------------------------------

        training_config = SFTConfig(

            output_dir=(
                f"/content/codemigrate_training/"
                f"{candidate_name}"
            ),

            max_length=MAX_LENGTH,

            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,

            gradient_accumulation_steps=4,

            max_steps=TRAIN_STEPS,

            learning_rate=params["learning_rate"],
            weight_decay=params["weight_decay"],

            warmup_steps=warmup_steps,

            lr_scheduler_type=params["scheduler"],

            optim="paged_adamw_8bit",

            bf16=True,
            gradient_checkpointing=True,

            completion_only_loss=True,
            packing=False,

            eval_strategy="no",

            logging_steps=4,
            save_strategy="no",
            report_to="none",

            dataloader_num_workers=0,

            seed=SEED,
            data_seed=SEED
        )

        # ------------------------------------------
        # Initialize trainer
        # ------------------------------------------

        trainer = SFTTrainer(
            model=candidate_model,
            args=training_config,
            train_dataset=optuna_train_dataset,
            eval_dataset=optuna_val_dataset,
            processing_class=tokenizer
        )

        # ------------------------------------------
        # Train
        # ------------------------------------------

        print(
            f"\n--- Training: {candidate_name} ---"
        )

        train_result = trainer.train()

        # ------------------------------------------
        # Validate
        # ------------------------------------------

        eval_result = trainer.evaluate()

        # ------------------------------------------
        # Save adapter
        # ------------------------------------------

        adapter_path = (
            ADAPTER_ROOT / candidate_name
        )

        candidate_model.save_pretrained(
            str(adapter_path),
            safe_serialization=True
        )

        tokenizer.save_pretrained(
            str(adapter_path)
        )

        elapsed = (
            time.perf_counter() - start_time
        )

        result = {
            "candidate": candidate_name,
            "path": str(adapter_path),
            "dropout": dropout,
            "train_loss": float(
                train_result.training_loss
            ),
            "validation_loss": float(
                eval_result["eval_loss"]
            ),
            "training_seconds": elapsed
        }

        print(
            f"\n--- {candidate_name} Results ---"
        )

        print(
            "Training loss:",
            round(result["train_loss"], 6)
        )

        print(
            "Validation loss:",
            round(result["validation_loss"], 6)
        )

        print(
            "Training time:",
            round(elapsed, 2),
            "seconds"
        )

        print(
            "Adapter saved:",
            adapter_path
        )

        return result

    finally:

        # ------------------------------------------
        # Remove trained adapter
        # ------------------------------------------

        if trainer is not None:
            del trainer

        if candidate_model is not None:

            base_model = candidate_model.unload()

            del candidate_model

        gc.collect()
        torch.cuda.empty_cache()


# --------------------------------------------------
# 3. Train Both Candidates
# --------------------------------------------------

candidate_results = {}

for name, params in CANDIDATES.items():

    candidate_results[name] = (
        train_and_save_candidate(
            candidate_name=name,
            params=params
        )
    )


# --------------------------------------------------
# 4. Compare Results
# --------------------------------------------------

print("\n--- Candidate Comparison ---")

for name, result in candidate_results.items():

    print(
        f"{name} | "
        f"Dropout={result['dropout']:.2f} | "
        f"Train Loss={result['train_loss']:.6f} | "
        f"Val Loss={result['validation_loss']:.6f}"
    )


# --------------------------------------------------
# 5. Verify Frozen Base Model
# --------------------------------------------------

remaining_lora_layers = sum(
    "lora_A" in name or "lora_B" in name
    for name, _ in base_model.named_parameters()
)

assert remaining_lora_layers == 0

print("\nBase model remained clean: PASS")
print("Both candidate adapters saved: PASS")

# Base vs Fine-Tuned CodeLlama:Compare Code Generation

In [ ]:
import gc
import torch

from pathlib import Path
from peft import PeftModel


# --------------------------------------------------
# 1. Unseen Migration Example
# --------------------------------------------------

test_example = {

    "migration_id": "test_feature_gate",

    "group_id": "test_feature_gate",

    "instruction": (
        "Migrate ConfigurationManager.AppSettings usage "
        "to modern .NET IConfiguration using constructor "
        "injection. Preserve the boolean return type and "
        "case-insensitive string comparison."
    ),

    "legacy_code": """
using System;
using System.Configuration;

public class FeatureGate
{
    public bool ShouldUseNewCheckout()
    {
        string value =
            ConfigurationManager.AppSettings["UseNewCheckout"];

        return string.Equals(
            value,
            "enabled",
            StringComparison.OrdinalIgnoreCase
        );
    }
}
""",

    "modern_code": """
using System;
using Microsoft.Extensions.Configuration;

public class FeatureGate
{
    private readonly IConfiguration _configuration;

    public FeatureGate(IConfiguration configuration)
    {
        _configuration = configuration;
    }

    public bool ShouldUseNewCheckout()
    {
        string? value =
            _configuration["UseNewCheckout"];

        return string.Equals(
            value,
            "enabled",
            StringComparison.OrdinalIgnoreCase
        );
    }
}
"""
}


# Verify this exact source was not used previously
previous_examples = (
    training_pool.to_list()
    + validation_candidates.to_list()
)

assert all(
    test_example["legacy_code"].strip()
    != row["legacy_code"].strip()
    for row in previous_examples
)


# --------------------------------------------------
# 2. Build CodeLlama Prompt
# --------------------------------------------------

test_prompt = prepare_sft_example(
    test_example
)["prompt"]


print("--- Test Query ---")
print(test_example["instruction"])

print("\n--- Legacy Code ---")
print(test_example["legacy_code"])


# --------------------------------------------------
# 3. Generation Function
# --------------------------------------------------

def generate_migration_code(
    model_name,
    adapter_path=None,
    max_new_tokens=256
):

    global base_model

    inference_model = None

    try:

        if adapter_path is None:

            # Original CodeLlama
            inference_model = base_model

        else:

            # Load fine-tuned LoRA adapter
            inference_model = PeftModel.from_pretrained(
                base_model,
                adapter_path,
                is_trainable=False
            )

        inference_model.eval()

        inputs = tokenizer(
            test_prompt,
            return_tensors="pt",
            add_special_tokens=False
        )

        device = next(
            inference_model.parameters()
        ).device

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        # ------------------------------------------
        # Generate without seeing the target code
        # ------------------------------------------

        with torch.inference_mode():

            outputs = inference_model.generate(
                **inputs,

                max_new_tokens=max_new_tokens,

                do_sample=False,
                num_beams=1,

                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,

                use_cache=True
            )

        # Extract only newly generated tokens
        generated_ids = outputs[
            0,
            inputs["input_ids"].shape[1]:
        ]

        generated_code = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

        return generated_code

    finally:

        # Remove LoRA adapter after inference
        if (
            adapter_path is not None
            and inference_model is not None
        ):

            base_model = inference_model.unload()

            del inference_model

        gc.collect()
        torch.cuda.empty_cache()


# --------------------------------------------------
# 4. Model Configurations
# --------------------------------------------------

models_to_compare = {

    "base_codellama": None,

    "optuna_best": (
        "/content/codemigrate_adapters/optuna_best"
    ),

    "mc_dropout": (
        "/content/codemigrate_adapters/mc_dropout"
    )
}


# Verify saved adapters exist
for name, path in models_to_compare.items():

    if path is not None:

        assert (
            Path(path) / "adapter_config.json"
        ).exists(), f"Adapter not found: {name}"


# --------------------------------------------------
# 5. Generate Predictions
# --------------------------------------------------

generation_results = {}

for model_name, adapter_path in models_to_compare.items():

    print("\n" + "=" * 65)
    print("MODEL:", model_name)

    generated_code = generate_migration_code(
        model_name=model_name,
        adapter_path=adapter_path
    )

    generation_results[model_name] = generated_code

    print("\nGenerated C# Code:\n")
    print(generated_code)


# --------------------------------------------------
# 6. Show Reference Migration
# --------------------------------------------------

print("\n" + "=" * 65)
print("REFERENCE MODERN C# CODE:\n")

print(test_example["modern_code"])


# --------------------------------------------------
# 7. Verify Base Model
# --------------------------------------------------

assert not any(
    "lora_A" in name or "lora_B" in name
    for name, _ in base_model.named_parameters()
)

print("\nInference comparison: PASS")
print("Base model remained clean: PASS")

# MC Dropout uncertainty analysis

In [ ]:
import gc
import numpy as np
import torch

from torch import nn
from peft import PeftModel


# ============================================================
# 1. Configuration
# ============================================================

MC_ADAPTER_PATH = "/content/codemigrate_adapters/mc_dropout"
MC_SAMPLES = 30

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


# ============================================================
# 2. Evaluation Example
# ============================================================

mc_test_example = {
    "migration_id": "test_feature_gate_mc",
    "group_id": "test_feature_gate",

    "instruction": (
        "Migrate ConfigurationManager.AppSettings usage to modern "
        ".NET IConfiguration using constructor injection. "
        "Preserve the boolean return type and case-insensitive "
        "string comparison."
    ),

    "legacy_code": """
using System;
using System.Configuration;

public class FeatureGate
{
    public bool ShouldUseNewCheckout()
    {
        string value =
            ConfigurationManager.AppSettings["UseNewCheckout"];

        return string.Equals(
            value,
            "enabled",
            StringComparison.OrdinalIgnoreCase
        );
    }
}
""",

    "modern_code": """
using System;
using Microsoft.Extensions.Configuration;

public class FeatureGate
{
    private readonly IConfiguration _configuration;

    public FeatureGate(IConfiguration configuration)
    {
        _configuration = configuration;
    }

    public bool ShouldUseNewCheckout()
    {
        string? value =
            _configuration["UseNewCheckout"];

        return string.Equals(
            value,
            "enabled",
            StringComparison.OrdinalIgnoreCase
        );
    }
}
"""
}


# ============================================================
# 3. Prepare Prompt and Reference Tokens
# ============================================================

mc_sft_example = prepare_sft_example(mc_test_example)

prompt_text = mc_sft_example["prompt"]
completion_text = mc_sft_example["completion"]

# The fast tokenizer provides exact character-to-token offsets.
# This prevents incorrect prompt/completion boundary alignment.

if getattr(tokenizer, "is_fast", False):

    encoding = tokenizer(
        prompt_text + completion_text,
        add_special_tokens=False,
        return_offsets_mapping=True
    )

    token_ids = encoding["input_ids"]
    offsets = encoding["offset_mapping"]

    prompt_end_char = len(prompt_text)

    completion_start = next(
        index
        for index, (start, end) in enumerate(offsets)
        if end > prompt_end_char
    )

else:

    # Fallback for slow tokenizers.
    prompt_ids = tokenizer.encode(
        prompt_text,
        add_special_tokens=False
    )

    completion_ids = tokenizer.encode(
        completion_text,
        add_special_tokens=False
    )

    token_ids = prompt_ids + completion_ids
    completion_start = len(prompt_ids)


assert completion_start > 0

device = next(base_model.parameters()).device

input_ids = torch.tensor(
    [token_ids],
    dtype=torch.long,
    device=device
)

target_ids = input_ids[0, completion_start:]

assert target_ids.numel() > 0

print("--- MC Dropout Input ---")
print("Total tokens:", input_ids.shape[1])
print("Reference tokens:", target_ids.numel())


# ============================================================
# 4. Reference Token Scoring Function
# ============================================================

def score_reference_tokens(model):
    """
    Computes teacher-forced probabilities for the reference
    modern C# code.

    Returns:
        mean_nll: Mean negative log-likelihood
        probabilities: Full vocabulary probabilities per token
        entropies: Predictive entropy per token
    """

    with torch.inference_mode():

        outputs = model(
            input_ids=input_ids,
            use_cache=False
        )

        # For causal language models:
        # logits at position t predict the token at t+1.

        logits = outputs.logits[
            0,
            completion_start - 1:-1,
            :
        ].float()

        assert logits.shape[0] == target_ids.numel()

        log_probs = torch.log_softmax(
            logits,
            dim=-1
        )

        probabilities = log_probs.exp()

        target_log_probs = log_probs.gather(
            dim=-1,
            index=target_ids.unsqueeze(-1)
        ).squeeze(-1)

        mean_nll = -target_log_probs.mean().item()

        entropies = -(
            probabilities * log_probs
        ).sum(dim=-1)

        return (
            mean_nll,
            probabilities.cpu(),
            entropies.cpu()
        )


# ============================================================
# 5. Load MC Dropout Adapter
# ============================================================

mc_model = PeftModel.from_pretrained(
    base_model,
    MC_ADAPTER_PATH,
    is_trainable=False
)

mc_model.eval()

print("\n--- Adapter Information ---")
print("Adapter:", MC_ADAPTER_PATH)


try:

    # ========================================================
    # 6. Deterministic Evaluation
    # ========================================================

    deterministic_nll, p_det, h_det = (
        score_reference_tokens(mc_model)
    )

    del p_det, h_det

    print("\n--- Deterministic Prediction ---")
    print(
        "Reference NLL:",
        round(deterministic_nll, 6)
    )


    # ========================================================
    # 7. Enable ONLY LoRA Dropout
    # ========================================================

    # Keep the full model in evaluation mode.
    # Activate only LoRA dropout modules with p > 0.

    active_dropout_layers = []

    for name, module in mc_model.named_modules():

        if (
            "lora_dropout" in name
            and isinstance(module, nn.Dropout)
            and module.p > 0
        ):

            module.train()

            active_dropout_layers.append(name)


    assert len(active_dropout_layers) > 0, (
        "No active LoRA dropout layers were found."
    )

    print("\n--- Dropout Configuration ---")
    print(
        "Active LoRA dropout layers:",
        len(active_dropout_layers)
    )


    # ========================================================
    # 8. Monte Carlo Forward Passes
    # ========================================================

    nll_samples = []

    probability_sum = None
    entropy_sum = None

    print("\n--- Running MC Dropout ---")

    for iteration in range(MC_SAMPLES):

        nll, probabilities, entropies = (
            score_reference_tokens(mc_model)
        )

        nll_samples.append(nll)

        if probability_sum is None:

            probability_sum = torch.zeros_like(
                probabilities,
                dtype=torch.float64
            )

            entropy_sum = torch.zeros_like(
                entropies,
                dtype=torch.float64
            )

        probability_sum += probabilities.double()

        entropy_sum += entropies.double()

        if (
            (iteration + 1) % 5 == 0
            or iteration == 0
        ):

            print(
                f"Pass {iteration + 1:02d}/{MC_SAMPLES} "
                f"| Reference NLL: {nll:.6f}"
            )


    # ========================================================
    # 9. MC Dropout Statistics
    # ========================================================

    nll_samples = np.asarray(
        nll_samples,
        dtype=np.float64
    )

    mean_nll = nll_samples.mean()

    std_nll = nll_samples.std(ddof=1)

    mean_probabilities = (
        probability_sum / MC_SAMPLES
    )

    mean_entropies = (
        entropy_sum / MC_SAMPLES
    )


    # ========================================================
    # 10. Predictive Entropy
    # ========================================================

    predictive_entropy = -(
        mean_probabilities
        * mean_probabilities.clamp_min(1e-12).log()
    ).sum(dim=-1)


    # ========================================================
    # 11. Epistemic Uncertainty
    # ========================================================

    # Mutual information:
    #
    # H(mean prediction) - mean(H(individual prediction))
    #
    # Higher values indicate greater disagreement among
    # dropout-based predictions.

    mutual_information = (
        predictive_entropy - mean_entropies
    ).clamp_min(0.0)

    mean_mutual_information = (
        mutual_information.mean().item()
    )


    # ========================================================
    # 12. Results
    # ========================================================

    print("\n" + "=" * 55)
    print("MC DROPOUT RESULTS")
    print("=" * 55)

    print(
        f"MC samples: {MC_SAMPLES}"
    )

    print(
        f"Deterministic reference NLL: "
        f"{deterministic_nll:.6f}"
    )

    print(
        f"MC mean reference NLL: "
        f"{mean_nll:.6f}"
    )

    print(
        f"MC reference NLL std: "
        f"{std_nll:.6f}"
    )

    print(
        f"Mean predictive entropy: "
        f"{predictive_entropy.mean().item():.6f}"
    )

    print(
        f"Mean epistemic uncertainty (MI): "
        f"{mean_mutual_information:.6f}"
    )


    # ========================================================
    # 13. Most Uncertain Reference Token Positions
    # ========================================================

    top_k = min(
        8,
        mutual_information.numel()
    )

    uncertain_positions = torch.topk(
        mutual_information,
        k=top_k
    ).indices.tolist()

    print("\n--- Most Uncertain Token Positions ---")

    for rank, position in enumerate(
        uncertain_positions,
        start=1
    ):

        token_id = target_ids[position].item()

        token_text = tokenizer.decode(
            [token_id]
        )

        uncertainty = (
            mutual_information[position].item()
        )

        print(
            f"{rank}. "
            f"Position={position} | "
            f"Token={repr(token_text)} | "
            f"MI={uncertainty:.6f}"
        )

    print("\nMC Dropout evaluation: PASS")


finally:

    # ========================================================
    # 14. Restore Clean Base Model
    # ========================================================

    mc_model.eval()

    base_model = mc_model.unload()

    del mc_model

    gc.collect()
    torch.cuda.empty_cache()

    remaining_lora_parameters = sum(
        "lora_A" in name or "lora_B" in name
        for name, _ in base_model.named_parameters()
    )

    assert remaining_lora_parameters == 0

    print("Base model restoration: PASS")

# Save Experiment Artifacts

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata as metadata
import shutil
import json
import uuid


# ============================================================
# 1. Mount Google Drive
# ============================================================

drive.mount("/content/drive")


# ============================================================
# 2. Create Versioned Experiment Directory
# ============================================================

RUN_ID = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_"
    + uuid.uuid4().hex[:6]
)

EXPERIMENT_DIR = (
    Path("/content/drive/MyDrive")
    / "CodeMigrateAI"
    / "experiments"
    / f"codellama_qlora_v3_{RUN_ID}"
)

EXPERIMENT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

print("Experiment directory:")
print(EXPERIMENT_DIR)


# ============================================================
# 3. Save QLoRA Adapters
# ============================================================

adapter_paths = {
    "optuna_best": Path(
        "/content/codemigrate_adapters/optuna_best"
    ),
    "mc_dropout": Path(
        "/content/codemigrate_adapters/mc_dropout"
    )
}

for adapter_name, source_path in adapter_paths.items():

    if not source_path.is_dir():
        raise FileNotFoundError(
            f"Adapter not found: {source_path}"
        )

    target_path = (
        EXPERIMENT_DIR
        / "adapters"
        / adapter_name
    )

    shutil.copytree(
        source_path,
        target_path
    )

    print(f"Saved adapter: {adapter_name}")


# ============================================================
# 4. Save Tokenizer
# ============================================================

tokenizer_path = EXPERIMENT_DIR / "tokenizer"

tokenizer.save_pretrained(
    str(tokenizer_path)
)

print("Tokenizer saved")


# ============================================================
# 5. Save Optuna Database
# ============================================================

optuna_db = Path(
    "/content/codemigrate_optuna_pilot.db"
)

if optuna_db.is_file():

    shutil.copy2(
        optuna_db,
        EXPERIMENT_DIR / "optuna_study.db"
    )

    print("Optuna database saved")

else:

    print("WARNING: Optuna database not found")


# ============================================================
# 6. Save Training and Validation Data
# ============================================================

data_dir = EXPERIMENT_DIR / "data"

data_dir.mkdir(exist_ok=True)

optuna_train_dataset.to_json(
    str(data_dir / "train.jsonl"),
    lines=True
)

validation_candidates.to_json(
    str(data_dir / "validation.jsonl"),
    lines=True
)

print("Training and validation datasets saved")


# ============================================================
# 7. Save Experiment Metadata
# ============================================================

package_names = [
    "transformers",
    "peft",
    "trl",
    "accelerate",
    "bitsandbytes",
    "datasets",
    "optuna"
]

package_versions = {}

for package_name in package_names:

    try:
        package_versions[package_name] = (
            metadata.version(package_name)
        )

    except metadata.PackageNotFoundError:
        package_versions[package_name] = None


experiment_metadata = {
    "project": "CodeMigrate AI",

    "model_id": MODEL_ID,

    "training_method": "QLoRA",

    "quantization": "4-bit",

    "optuna_study": "codemigrate_qlora_pilot_v3",

    "best_trial": best_trial.number,

    "best_hyperparameters": best_trial.params,

    "best_optuna_validation_loss": best_trial.value,

    "final_adapters": {
        "optuna_best": {
            "dropout": 0.0,
            "validation_loss": 0.213394
        },

        "mc_dropout": {
            "dropout": 0.10,
            "validation_loss": 0.214618
        }
    },

    "mc_dropout_results": {
        "samples": MC_SAMPLES,
        "deterministic_nll": deterministic_nll,
        "mean_nll": float(mean_nll),
        "std_nll": float(std_nll),
        "mean_mutual_information": float(
            mean_mutual_information
        )
    },

    "dataset_sizes": {
        "train": len(optuna_train_dataset),
        "validation": len(validation_candidates)
    },

    "evaluation_status": (
        "Pilot evaluation only; "
        "independent held-out test pending"
    ),

    "library_versions": package_versions
}


metadata_path = (
    EXPERIMENT_DIR
    / "experiment_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        experiment_metadata,
        file,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 8. Verify Saved Files
# ============================================================

required_paths = [
    EXPERIMENT_DIR / "adapters" / "optuna_best",
    EXPERIMENT_DIR / "adapters" / "mc_dropout",
    EXPERIMENT_DIR / "tokenizer",
    data_dir / "train.jsonl",
    data_dir / "validation.jsonl",
    metadata_path
]

for path in required_paths:
    assert path.exists(), f"Missing artifact: {path}"


print("\n" + "=" * 55)
print("EXPERIMENT BACKUP")
print("=" * 55)

print("Model:", MODEL_ID)
print("Best Optuna trial:", best_trial.number)
print("Saved adapters:", len(adapter_paths))
print("Training examples:", len(optuna_train_dataset))
print("Validation examples:", len(validation_candidates))

print("\nSaved to:")
print(EXPERIMENT_DIR)

print("\nExperiment backup: PASS")

# Independent Test Set

In [ ]:
from datasets import Dataset
from collections import Counter


# ============================================================
# 1. Helper Function
# ============================================================

def make_test_case(
    migration_id,
    group_id,
    instruction,
    legacy_code,
    modern_code
):
    return {
        "migration_id": migration_id,
        "group_id": group_id,
        "instruction": instruction.strip(),
        "legacy_code": legacy_code.strip(),
        "modern_code": modern_code.strip()
    }


# ============================================================
# 2. Independent Migration Test Cases
# ============================================================

test_cases = [

    # --------------------------------------------------------
    # TEST 1: Configuration with Default Value
    # --------------------------------------------------------

    make_test_case(
        "test_config_default",
        "test_configuration",

        """
        Replace ConfigurationManager.AppSettings with
        IConfiguration constructor injection.
        Preserve the default value when the key is missing.
        """,

        """
using System.Configuration;

public class RegionSettings
{
    public string GetRegion()
    {
        return ConfigurationManager.AppSettings["Region"]
               ?? "global";
    }
}
""",

        """
using Microsoft.Extensions.Configuration;

public class RegionSettings
{
    private readonly IConfiguration _configuration;

    public RegionSettings(IConfiguration configuration)
    {
        _configuration = configuration;
    }

    public string GetRegion()
    {
        return _configuration["Region"] ?? "global";
    }
}
"""
    ),


    # --------------------------------------------------------
    # TEST 2: Connection String Migration
    # --------------------------------------------------------

    make_test_case(
        "test_connection_string",
        "test_configuration",

        """
        Migrate ConfigurationManager.ConnectionStrings
        to IConfiguration.GetConnectionString.
        Use constructor injection and preserve the fallback.
        """,

        """
using System.Configuration;

public class DatabaseSettings
{
    public string GetConnection()
    {
        return ConfigurationManager
            .ConnectionStrings["MainDatabase"]?.ConnectionString
            ?? "";
    }
}
""",

        """
using Microsoft.Extensions.Configuration;

public class DatabaseSettings
{
    private readonly IConfiguration _configuration;

    public DatabaseSettings(IConfiguration configuration)
    {
        _configuration = configuration;
    }

    public string GetConnection()
    {
        return _configuration.GetConnectionString("MainDatabase")
               ?? "";
    }
}
"""
    ),


    # --------------------------------------------------------
    # TEST 3: HTTPS Detection
    # --------------------------------------------------------

    make_test_case(
        "test_https",
        "test_http_context",

        """
        Migrate ASP.NET MVC 5 request HTTPS detection
        to ASP.NET Core. Preserve the boolean result.
        """,

        """
using System.Web.Mvc;

public class SecurityController : Controller
{
    public bool IsSecure()
    {
        return Request.IsSecureConnection;
    }
}
""",

        """
using Microsoft.AspNetCore.Mvc;

public class SecurityController : Controller
{
    public bool IsSecure()
    {
        return Request.IsHttps;
    }
}
"""
    ),


    # --------------------------------------------------------
    # TEST 4: HTTP Method
    # --------------------------------------------------------

    make_test_case(
        "test_http_method",
        "test_http_context",

        """
        Migrate ASP.NET MVC 5 request method access
        to ASP.NET Core. Preserve the string return type.
        """,

        """
using System.Web.Mvc;

public class RequestInfoController : Controller
{
    public string GetMethod()
    {
        return Request.HttpMethod;
    }
}
""",

        """
using Microsoft.AspNetCore.Mvc;

public class RequestInfoController : Controller
{
    public string GetMethod()
    {
        return Request.Method;
    }
}
"""
    ),


    # --------------------------------------------------------
    # TEST 5: Client IP Address
    # --------------------------------------------------------

    make_test_case(
        "test_client_ip",
        "test_http_context",

        """
        Migrate legacy ASP.NET request client IP access
        to ASP.NET Core HttpContext.Connection.
        Preserve nullable behavior.
        """,

        """
using System.Web.Mvc;

public class NetworkController : Controller
{
    public string GetClientIp()
    {
        return Request.UserHostAddress;
    }
}
""",

        """
using Microsoft.AspNetCore.Mvc;

public class NetworkController : Controller
{
    public string? GetClientIp()
    {
        return HttpContext.Connection.RemoteIpAddress?.ToString();
    }
}
"""
    ),


    # --------------------------------------------------------
    # TEST 6: HttpNotFound Migration
    # --------------------------------------------------------

    make_test_case(
        "test_http_not_found",
        "test_mvc",

        """
        Migrate ASP.NET MVC 5 HttpNotFound to ASP.NET Core.
        Preserve the 404 response and successful content.
        """,

        """
using System.Web.Mvc;

public class ProductController : Controller
{
    public ActionResult Details(int id)
    {
        if (id <= 0)
            return HttpNotFound();

        return Content(id.ToString());
    }
}
""",

        """
using Microsoft.AspNetCore.Mvc;

public class ProductController : Controller
{
    public IActionResult Details(int id)
    {
        if (id <= 0)
            return NotFound();

        return Content(id.ToString());
    }
}
"""
    ),


    # --------------------------------------------------------
    # TEST 7: HTTP Status Code
    # --------------------------------------------------------

    make_test_case(
        "test_http_status",
        "test_mvc",

        """
        Migrate HttpStatusCodeResult from ASP.NET MVC 5
        to ASP.NET Core. Preserve HTTP status code 403.
        """,

        """
using System.Web.Mvc;

public class AccessController : Controller
{
    public ActionResult Denied()
    {
        return new HttpStatusCodeResult(403);
    }
}
""",

        """
using Microsoft.AspNetCore.Mvc;

public class AccessController : Controller
{
    public IActionResult Denied()
    {
        return StatusCode(403);
    }
}
"""
    ),


    # --------------------------------------------------------
    # TEST 8: EF6 Any to EF Core AnyAsync
    # --------------------------------------------------------

    make_test_case(
        "test_ef_any",
        "test_entity_framework",

        """
        Migrate a synchronous Entity Framework 6 Any query
        to asynchronous EF Core AnyAsync.
        Preserve the predicate and boolean result.
        """,

        """
using System.Linq;

public class OrderQueries
{
    private readonly StoreContext db;

    public OrderQueries(StoreContext context)
    {
        db = context;
    }

    public bool HasOpenOrders()
    {
        return db.Orders.Any(o => o.IsOpen);
    }
}
""",

        """
using System.Threading.Tasks;
using Microsoft.EntityFrameworkCore;

public class OrderQueries
{
    private readonly StoreContext db;

    public OrderQueries(StoreContext context)
    {
        db = context;
    }

    public async Task<bool> HasOpenOrders()
    {
        return await db.Orders.AnyAsync(o => o.IsOpen);
    }
}
"""
    ),


    # --------------------------------------------------------
    # TEST 9: EF6 FirstOrDefault to EF Core Async
    # --------------------------------------------------------

    make_test_case(
        "test_ef_first",
        "test_entity_framework",

        """
        Migrate an EF6 synchronous FirstOrDefault query
        to EF Core FirstOrDefaultAsync.
        Preserve the lookup predicate and null behavior.
        """,

        """
using System.Linq;

public class ProductQueries
{
    private readonly StoreContext db;

    public ProductQueries(StoreContext context)
    {
        db = context;
    }

    public Product GetById(int id)
    {
        return db.Products.FirstOrDefault(p => p.Id == id);
    }
}
""",

        """
using System.Threading.Tasks;
using Microsoft.EntityFrameworkCore;

public class ProductQueries
{
    private readonly StoreContext db;

    public ProductQueries(StoreContext context)
    {
        db = context;
    }

    public async Task<Product?> GetById(int id)
    {
        return await db.Products
            .FirstOrDefaultAsync(p => p.Id == id);
    }
}
"""
    ),


    # --------------------------------------------------------
    # TEST 10: EF6 ToList to EF Core ToListAsync
    # --------------------------------------------------------

    make_test_case(
        "test_ef_list",
        "test_entity_framework",

        """
        Migrate an EF6 synchronous ordered ToList query
        to asynchronous EF Core ToListAsync.
        Preserve the ordering and returned items.
        """,

        """
using System.Collections.Generic;
using System.Linq;

public class CustomerQueries
{
    private readonly StoreContext db;

    public CustomerQueries(StoreContext context)
    {
        db = context;
    }

    public List<Customer> GetCustomers()
    {
        return db.Customers
            .OrderBy(c => c.Name)
            .ToList();
    }
}
""",

        """
using System.Collections.Generic;
using System.Linq;
using System.Threading.Tasks;
using Microsoft.EntityFrameworkCore;

public class CustomerQueries
{
    private readonly StoreContext db;

    public CustomerQueries(StoreContext context)
    {
        db = context;
    }

    public async Task<List<Customer>> GetCustomers()
    {
        return await db.Customers
            .OrderBy(c => c.Name)
            .ToListAsync();
    }
}
"""
    )
]


# ============================================================
# 3. Create Hugging Face Dataset
# ============================================================

heldout_test_dataset = Dataset.from_list(test_cases)


# ============================================================
# 4. Data Integrity Checks
# ============================================================

assert len(heldout_test_dataset) == 10

migration_ids = heldout_test_dataset["migration_id"]

assert len(migration_ids) == len(set(migration_ids))

for example in heldout_test_dataset:

    assert example["instruction"].strip()
    assert example["legacy_code"].strip()
    assert example["modern_code"].strip()

    assert (
        example["legacy_code"].strip()
        != example["modern_code"].strip()
    )


# ============================================================
# 5. Check Exact Prompt Overlap
# ============================================================

heldout_sft_dataset = heldout_test_dataset.map(
    prepare_sft_example,
    remove_columns=heldout_test_dataset.column_names
)


def normalize_text(text):
    return " ".join(text.split())


train_prompts = {
    normalize_text(example["prompt"])
    for example in optuna_train_dataset
}

validation_prompts = {
    normalize_text(example["prompt"])
    for example in optuna_val_dataset
}

test_prompts = {
    normalize_text(example["prompt"])
    for example in heldout_sft_dataset
}


train_overlap = train_prompts.intersection(test_prompts)

validation_overlap = validation_prompts.intersection(test_prompts)

assert len(train_overlap) == 0
assert len(validation_overlap) == 0

assert len(test_prompts) == len(heldout_test_dataset)


# ============================================================
# 6. Check Sequence Lengths
# ============================================================

token_lengths = []

for example in heldout_sft_dataset:

    token_count = len(
        tokenizer.encode(
            example["prompt"] + example["completion"],
            add_special_tokens=False
        )
    )

    token_lengths.append(token_count)


print("--- Independent Test Dataset ---")

print("Total examples:", len(heldout_test_dataset))

print("\nGroup distribution:")
print(Counter(heldout_test_dataset["group_id"]))

print("\nToken length statistics:")
print("Minimum:", min(token_lengths))
print("Median:", int(sorted(token_lengths)[len(token_lengths) // 2]))
print("Maximum:", max(token_lengths))

print("\n--- Leakage Checks ---")
print("Exact train prompt overlap:", len(train_overlap))
print("Exact validation prompt overlap:", len(validation_overlap))

print("\n--- Migration Scenarios ---")

for example in heldout_test_dataset:

    print(
        example["migration_id"],
        "| Group:",
        example["group_id"]
    )

print("\nIndependent test dataset preparation: PASS")

# Base CodeLlama vs QLoRA Models

In [ ]:
import gc
import json
import time
from pathlib import Path

import torch

from peft import PeftModel


# ============================================================
# 1. Experiment Configuration
# ============================================================

MAX_NEW_TOKENS = 384

MODEL_CONFIGS = {
    "base_codellama": None,

    "optuna_best": (
        "/content/codemigrate_adapters/optuna_best"
    ),

    "mc_dropout": (
        "/content/codemigrate_adapters/mc_dropout"
    )
}

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


# ============================================================
# 2. Output Directory
# ============================================================

evaluation_dir = Path(EXPERIMENT_DIR) / "evaluation"

evaluation_dir.mkdir(
    parents=True,
    exist_ok=True
)

test_dataset_path = (
    evaluation_dir / "heldout_test_v1.jsonl"
)

results_path = (
    evaluation_dir / "generation_results_v1.jsonl"
)


# ============================================================
# 3. Freeze Independent Test Dataset
# ============================================================

if not test_dataset_path.exists():

    heldout_test_dataset.to_json(
        str(test_dataset_path),
        lines=True
    )

    print("Held-out test dataset saved")

else:

    with open(
        test_dataset_path,
        "r",
        encoding="utf-8"
    ) as file:

        saved_test_cases = [
            json.loads(line)
            for line in file
            if line.strip()
        ]

    assert saved_test_cases == list(
        heldout_test_dataset
    ), "Saved test dataset differs from current dataset"

    print("Existing held-out test dataset verified")


# ============================================================
# 4. Restore Previous Results If Available
# ============================================================

results = []

if results_path.exists():

    with open(
        results_path,
        "r",
        encoding="utf-8"
    ) as file:

        results = [
            json.loads(line)
            for line in file
            if line.strip()
        ]


completed_keys = {
    (
        result["model"],
        result["migration_id"]
    )
    for result in results
}

assert len(completed_keys) == len(results), (
    "Duplicate results found in checkpoint"
)

print("\nPreviously completed evaluations:", len(results))


# ============================================================
# 5. Generation Function
# ============================================================

def generate_migration(model, prompt):

    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(next(model.parameters()).device)

    prompt_length = inputs["input_ids"].shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    with torch.inference_mode():

        generated_ids = model.generate(
            **inputs,

            max_new_tokens=MAX_NEW_TOKENS,

            do_sample=False,
            num_beams=1,

            use_cache=True,

            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed_time = time.perf_counter() - start_time

    completion_ids = generated_ids[
        0,
        prompt_length:
    ]

    generated_code = tokenizer.decode(
        completion_ids,
        skip_special_tokens=True
    ).strip()

    stopped_by_eos = (
        len(completion_ids) > 0
        and completion_ids[-1].item()
        == tokenizer.eos_token_id
    )

    return {
        "generated_code": generated_code,
        "generation_seconds": elapsed_time,
        "generated_tokens": len(completion_ids),
        "stopped_by_eos": bool(stopped_by_eos)
    }


# ============================================================
# 6. Verify Base Model Is Clean
# ============================================================

remaining_lora = sum(
    "lora_A" in name or "lora_B" in name
    for name, _ in base_model.named_parameters()
)

assert remaining_lora == 0

print("\nBase model clean: PASS")


# ============================================================
# 7. Evaluate All Three Models
# ============================================================

for model_name, adapter_path in MODEL_CONFIGS.items():

    pending_indices = [
        index
        for index, example in enumerate(heldout_test_dataset)
        if (
            model_name,
            example["migration_id"]
        ) not in completed_keys
    ]

    print("\n" + "=" * 60)
    print("MODEL:", model_name)
    print("Remaining examples:", len(pending_indices))
    print("=" * 60)

    if len(pending_indices) == 0:
        print("Already completed. Skipping.")
        continue

    active_model = None

    try:

        # ----------------------------------------------------
        # Load Model
        # ----------------------------------------------------

        if adapter_path is None:

            active_model = base_model

        else:

            active_model = PeftModel.from_pretrained(
                base_model,
                adapter_path,
                is_trainable=False
            )

        active_model.eval()


        # ----------------------------------------------------
        # Generate All Pending Migrations
        # ----------------------------------------------------

        for index in pending_indices:

            example = heldout_test_dataset[index]

            prompt = heldout_sft_dataset[index]["prompt"]

            print(
                f"\n[{index + 1}/{len(heldout_test_dataset)}] "
                f"{example['migration_id']}"
            )

            generation = generate_migration(
                active_model,
                prompt
            )

            result = {
                "model": model_name,

                "migration_id": example["migration_id"],

                "group_id": example["group_id"],

                "instruction": example["instruction"],

                "legacy_code": example["legacy_code"],

                "reference_code": example["modern_code"],

                **generation
            }

            # ------------------------------------------------
            # Save Immediately
            # ------------------------------------------------

            with open(
                results_path,
                "a",
                encoding="utf-8"
            ) as file:

                file.write(
                    json.dumps(
                        result,
                        ensure_ascii=False
                    ) + "\n"
                )

            results.append(result)

            completed_keys.add(
                (
                    model_name,
                    example["migration_id"]
                )
            )

            print(
                f"Time: "
                f"{generation['generation_seconds']:.2f} sec"
            )

            print(
                "Generated tokens:",
                generation["generated_tokens"]
            )

            print(
                "Stopped by EOS:",
                generation["stopped_by_eos"]
            )

            preview = " | ".join(
                generation["generated_code"].splitlines()[:3]
            )

            print(
                "Preview:",
                preview[:160]
            )


    finally:

        # ----------------------------------------------------
        # Restore Clean Base Model
        # ----------------------------------------------------

        if adapter_path is not None and active_model is not None:

            active_model.eval()

            base_model = active_model.unload()

            del active_model

        gc.collect()
        torch.cuda.empty_cache()

        remaining_lora = sum(
            "lora_A" in name or "lora_B" in name
            for name, _ in base_model.named_parameters()
        )

        assert remaining_lora == 0

        print("\nBase model restoration: PASS")


# ============================================================
# 8. Final Result Verification
# ============================================================

expected_keys = {
    (model_name, example["migration_id"])
    for model_name in MODEL_CONFIGS
    for example in heldout_test_dataset
}

assert completed_keys == expected_keys

assert len(results) == len(expected_keys)


# ============================================================
# 9. Summary
# ============================================================

print("\n" + "=" * 60)
print("INDEPENDENT INFERENCE RESULTS")
print("=" * 60)

for model_name in MODEL_CONFIGS:

    model_results = [
        result
        for result in results
        if result["model"] == model_name
    ]

    total_time = sum(
        result["generation_seconds"]
        for result in model_results
    )

    avg_time = total_time / len(model_results)

    eos_count = sum(
        result["stopped_by_eos"]
        for result in model_results
    )

    print(f"\nModel: {model_name}")

    print(
        "Completed migrations:",
        len(model_results)
    )

    print(
        f"Average generation time: "
        f"{avg_time:.2f} seconds"
    )

    print(
        f"EOS completions: "
        f"{eos_count}/{len(model_results)}"
    )


print("\nTotal generated migrations:", len(results))

print("\nResults saved to:")
print(results_path)

print("\nIndependent inference: PASS")

In [ ]:
import re
import json
import pandas as pd

from pathlib import Path


# ============================================================
# 1. Load Generated Results
# ============================================================

results_path = (
    Path(EXPERIMENT_DIR)
    / "evaluation"
    / "generation_results_v1.jsonl"
)

with open(
    results_path,
    "r",
    encoding="utf-8"
) as file:

    generation_results = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

assert len(generation_results) == 30


# ============================================================
# 2. Define Migration Rules
# ============================================================

MIGRATION_RULES = {

    "test_config_default": {
        "required": [
            r"\bIConfiguration\b",
            r'\[\s*"Region"\s*\]',
            r'\?\?\s*"global"'
        ],
        "forbidden": [
            r"\bConfigurationManager\b"
        ]
    },

    "test_connection_string": {
        "required": [
            r"\bIConfiguration\b",
            r"\bGetConnectionString\s*\(",
            r'"MainDatabase"'
        ],
        "forbidden": [
            r"\bConfigurationManager\b"
        ]
    },

    "test_https": {
        "required": [
            r"\bRequest\.IsHttps\b"
        ],
        "forbidden": [
            r"\bIsSecureConnection\b"
        ]
    },

    "test_http_method": {
        "required": [
            r"\bRequest\.Method\b"
        ],
        "forbidden": [
            r"\bRequest\.HttpMethod\b"
        ]
    },

    "test_client_ip": {
        "required": [
            r"\bRemoteIpAddress\b",
            r"\bToString\s*\("
        ],
        "forbidden": [
            r"\bUserHostAddress\b"
        ]
    },

    "test_http_not_found": {
        "required": [
            r"\bNotFound\s*\(",
            r"\bContent\s*\("
        ],
        "forbidden": [
            r"\bHttpNotFound\s*\("
        ]
    },

    "test_http_status": {
        "required": [
            r"\bStatusCode\s*\(\s*403\s*\)"
        ],
        "forbidden": [
            r"\bHttpStatusCodeResult\b"
        ]
    },

    "test_ef_any": {
        "required": [
            r"\bAnyAsync\s*\(",
            r"\bIsOpen\b",
            r"\bTask\s*<\s*bool\s*>"
        ],
        "forbidden": [
            r"\.Any\s*\("
        ]
    },

    "test_ef_first": {
        "required": [
            r"\bFirstOrDefaultAsync\s*\(",
            r"\bTask\s*<",
            r"\bId\b"
        ],
        "forbidden": [
            r"\.FirstOrDefault\s*\("
        ]
    },

    "test_ef_list": {
        "required": [
            r"\bOrderBy\s*\(",
            r"\bToListAsync\s*\(",
            r"\bTask\s*<"
        ],
        "forbidden": [
            r"\.ToList\s*\("
        ]
    }
}


# ============================================================
# 3. Code-Only Output Check
# ============================================================

def check_code_only_output(text):

    code = text.strip()

    if not code:
        return False

    # Markdown code fences are forbidden by our prompt.
    if "```" in code:
        return False

    # Common explanation prefixes.
    explanation_patterns = [
        r"^\s*here is\b",
        r"^\s*the migrated code\b",
        r"^\s*explanation\s*:",
        r"^\s*note\s*:"
    ]

    for pattern in explanation_patterns:

        if re.search(
            pattern,
            code,
            flags=re.IGNORECASE
        ):
            return False

    # Expected C# starting constructs.
    code_start_pattern = (
        r"^\s*(?:"
        r"using\s+|"
        r"namespace\s+|"
        r"public\s+|"
        r"internal\s+|"
        r"#nullable\b|"
        r"\["
        r")"
    )

    return bool(
        re.search(code_start_pattern, code)
    )


# ============================================================
# 4. Migration Rule Check
# ============================================================

def check_migration_rules(migration_id, generated_code):

    rules = MIGRATION_RULES[migration_id]

    missing_required = []

    found_forbidden = []

    for pattern in rules["required"]:

        if not re.search(
            pattern,
            generated_code
        ):
            missing_required.append(pattern)

    for pattern in rules["forbidden"]:

        if re.search(
            pattern,
            generated_code
        ):
            found_forbidden.append(pattern)

    return {
        "required_rules_pass": (
            len(missing_required) == 0
        ),

        "legacy_api_absent": (
            len(found_forbidden) == 0
        ),

        "missing_required": missing_required,

        "found_forbidden": found_forbidden
    }


# ============================================================
# 5. Possible Missing MVC Namespace
# ============================================================

def check_mvc_namespace_warning(code):

    # Detect classes inheriting Controller / ControllerBase.
    inherits_mvc_controller = bool(
        re.search(
            r"\bclass\s+\w+\s*:\s*"
            r"(?:Controller|ControllerBase)\b",
            code
        )
    )

    if not inherits_mvc_controller:
        return False

    has_mvc_using = bool(
        re.search(
            r"\busing\s+Microsoft\.AspNetCore\.Mvc\s*;",
            code
        )
    )

    has_fully_qualified_type = bool(
        re.search(
            r":\s*Microsoft\.AspNetCore\.Mvc\."
            r"(?:Controller|ControllerBase)\b",
            code
        )
    )

    return not (
        has_mvc_using
        or has_fully_qualified_type
    )


# ============================================================
# 6. Evaluate All Generated Codes
# ============================================================

review_records = []

for result in generation_results:

    model_name = result["model"]

    migration_id = result["migration_id"]

    generated_code = result["generated_code"]

    rule_result = check_migration_rules(
        migration_id,
        generated_code
    )

    code_only = check_code_only_output(
        generated_code
    )

    mvc_warning = check_mvc_namespace_warning(
        generated_code
    )

    rule_signal_pass = (
        rule_result["required_rules_pass"]
        and rule_result["legacy_api_absent"]
    )

    needs_review = (
        not code_only
        or not rule_signal_pass
        or mvc_warning
        or not result["stopped_by_eos"]
    )

    review_records.append({

        "model": model_name,

        "migration_id": migration_id,

        "group_id": result["group_id"],

        "code_only": code_only,

        "required_rules_pass": (
            rule_result["required_rules_pass"]
        ),

        "legacy_api_absent": (
            rule_result["legacy_api_absent"]
        ),

        "rule_signal_pass": rule_signal_pass,

        "possible_missing_mvc_using": mvc_warning,

        "stopped_by_eos": result["stopped_by_eos"],

        "needs_review": needs_review,

        "missing_required": (
            ", ".join(rule_result["missing_required"])
        ),

        "found_forbidden": (
            ", ".join(rule_result["found_forbidden"])
        )

    })


# ============================================================
# 7. Create Results DataFrame
# ============================================================

review_df = pd.DataFrame(review_records)

model_order = [
    "base_codellama",
    "optuna_best",
    "mc_dropout"
]

review_df["model"] = pd.Categorical(
    review_df["model"],
    categories=model_order,
    ordered=True
)


# ============================================================
# 8. Aggregate Results
# ============================================================

summary_df = review_df.groupby(
    "model",
    observed=True
).agg(

    examples=("migration_id", "count"),

    code_only_rate=("code_only", "mean"),

    required_rule_rate=(
        "required_rules_pass",
        "mean"
    ),

    legacy_api_removed_rate=(
        "legacy_api_absent",
        "mean"
    ),

    rule_signal_rate=(
        "rule_signal_pass",
        "mean"
    ),

    mvc_namespace_warnings=(
        "possible_missing_mvc_using",
        "sum"
    ),

    eos_rate=("stopped_by_eos", "mean")

)


# ============================================================
# 9. Print Summary
# ============================================================

print("=" * 65)
print("AUTOMATED MIGRATION REVIEW")
print("=" * 65)

print("\n--- Model Comparison ---")

print(
    summary_df.round(4).to_string()
)


# ============================================================
# 10. Print Cases Requiring Review
# ============================================================

print("\n--- Cases Requiring Manual Review ---")

flagged = review_df[
    review_df["needs_review"]
]

if flagged.empty:

    print("No cases flagged")

else:

    for _, row in flagged.iterrows():

        print(
            f"\nModel: {row['model']} | "
            f"Case: {row['migration_id']}"
        )

        if not row["code_only"]:
            print("  - Code-only format failed")

        if not row["required_rules_pass"]:
            print(
                "  - Missing expected API patterns:",
                row["missing_required"]
            )

        if not row["legacy_api_absent"]:
            print(
                "  - Legacy API still present:",
                row["found_forbidden"]
            )

        if row["possible_missing_mvc_using"]:
            print(
                "  - Possible missing MVC namespace"
            )

        if not row["stopped_by_eos"]:
            print(
                "  - Generation did not stop at EOS"
            )


# ============================================================
# 11. Save Review Results
# ============================================================

review_path = (
    Path(EXPERIMENT_DIR)
    / "evaluation"
    / "static_review_v1.csv"
)

review_df.to_csv(
    review_path,
    index=False
)

assert review_path.exists()

print("\nReview results saved:")
print(review_path)

print("\nAutomated migration review: PASS")

**Reviewing suspicious migration outputs**

In [ ]:
import json
from pathlib import Path


# ============================================================
# 1. Load Generation Results
# ============================================================

results_file = (
    Path(EXPERIMENT_DIR)
    / "evaluation"
    / "generation_results_v1.jsonl"
)

with open(
    results_file,
    "r",
    encoding="utf-8"
) as file:

    generated_results = [
        json.loads(line)
        for line in file
        if line.strip()
    ]


# ============================================================
# 2. Select Cases Requiring Inspection
# ============================================================

focus_cases = [
    "test_connection_string",
    "test_https",
    "test_client_ip"
]

focus_models = [
    "optuna_best",
    "mc_dropout"
]


# ============================================================
# 3. Compare Generated Code with Reference
# ============================================================

for migration_id in focus_cases:

    matching_results = [
        result
        for result in generated_results
        if result["migration_id"] == migration_id
    ]

    assert len(matching_results) == 3

    reference_code = matching_results[0]["reference_code"]

    print("\n" + "=" * 75)
    print("MIGRATION:", migration_id)
    print("=" * 75)

    print("\n--- REFERENCE CODE ---")
    print(reference_code)

    for model_name in focus_models:

        result = next(
            item
            for item in matching_results
            if item["model"] == model_name
        )

        print("\n" + "-" * 65)
        print("MODEL:", model_name)
        print("-" * 65)

        print(result["generated_code"])

        print(
            "\nEOS completed:",
            result["stopped_by_eos"]
        )


# ============================================================
# 4. Compare Both QLoRA Model Outputs
# ============================================================

print("\n" + "=" * 75)
print("QLoRA OUTPUT AGREEMENT")
print("=" * 75)

agreement_count = 0

for example in heldout_test_dataset:

    migration_id = example["migration_id"]

    best_output = next(
        result["generated_code"]
        for result in generated_results
        if result["migration_id"] == migration_id
        and result["model"] == "optuna_best"
    )

    dropout_output = next(
        result["generated_code"]
        for result in generated_results
        if result["migration_id"] == migration_id
        and result["model"] == "mc_dropout"
    )

    identical = (
        best_output.strip() == dropout_output.strip()
    )

    agreement_count += int(identical)

    print(
        f"{migration_id}: "
        f"{'IDENTICAL' if identical else 'DIFFERENT'}"
    )


print("\nIdentical outputs:", agreement_count, "/10")
print("\nFocused code inspection: PASS")

# NET SDK Install

In [ ]:
import os
import shutil
import subprocess

from pathlib import Path


# ============================================================
# 1. Check Existing .NET SDK
# ============================================================

def get_installed_sdks(dotnet_executable):

    if dotnet_executable is None:
        return ""

    result = subprocess.run(
        [str(dotnet_executable), "--list-sdks"],
        capture_output=True,
        text=True,
        check=True
    )

    return result.stdout


dotnet_executable = shutil.which("dotnet")

installed_sdks = get_installed_sdks(
    dotnet_executable
)

print("--- Existing .NET SDKs ---")
print(installed_sdks or "No SDK found")


# ============================================================
# 2. Install .NET 10 SDK If Needed
# ============================================================

has_dotnet_10 = any(
    line.strip().startswith("10.")
    for line in installed_sdks.splitlines()
)

if not has_dotnet_10:

    print("\n.NET 10 SDK not found. Installing...")

    install_script = Path(
        "/content/dotnet-install.sh"
    )

    install_directory = Path(
        "/content/dotnet10"
    )

    install_directory.mkdir(
        parents=True,
        exist_ok=True
    )

    # Download Microsoft's official installation script.
    subprocess.run(
        [
            "curl",
            "-fL",
            "--retry", "3",
            "--output", str(install_script),
            "https://dot.net/v1/dotnet-install.sh"
        ],
        check=True
    )

    # Install the latest SDK from the .NET 10 channel.
    subprocess.run(
        [
            "bash",
            str(install_script),
            "--channel", "10.0",
            "--install-dir", str(install_directory)
        ],
        check=True
    )

    os.environ["DOTNET_ROOT"] = str(
        install_directory
    )

    os.environ["PATH"] = (
        str(install_directory)
        + os.pathsep
        + os.environ["PATH"]
    )

    dotnet_executable = str(
        install_directory / "dotnet"
    )

else:

    print("\n.NET 10 SDK already available")


# ============================================================
# 3. Verify Installation
# ============================================================

print("\n--- Final SDK Information ---")

result = subprocess.run(
    [dotnet_executable, "--list-sdks"],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout)

assert any(
    line.strip().startswith("10.")
    for line in result.stdout.splitlines()
)

print("\n.NET SDK installation: PASS")

# C# Compilation Test Harness

In [ ]:
import os
import shutil
import subprocess

from pathlib import Path


# ============================================================
# 1. Locate .NET SDK
# ============================================================

dotnet_path = Path("/content/dotnet10/dotnet")

if not dotnet_path.is_file():

    discovered = shutil.which("dotnet")

    if discovered is None:
        raise RuntimeError(
            ".NET SDK not found. Run Cell 34 first."
        )

    dotnet_path = Path(discovered)


print("--- .NET Compiler ---")
print("Executable:", dotnet_path)


# ============================================================
# 2. Create Compilation Test Directory
# ============================================================

COMPILE_DIR = Path(
    "/content/codemigrate_compile_tests"
)

COMPILE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PROJECT_FILE = COMPILE_DIR / "CodeMigrateCompile.csproj"

SOURCE_FILE = COMPILE_DIR / "Migration.cs"

TEST_TYPES_FILE = COMPILE_DIR / "TestTypes.cs"


# ============================================================
# 3. Create .NET Project
# ============================================================

project_configuration = """
<Project Sdk="Microsoft.NET.Sdk">

  <PropertyGroup>
    <TargetFramework>net10.0</TargetFramework>
    <LangVersion>latest</LangVersion>
    <Nullable>enable</Nullable>
    <ImplicitUsings>disable</ImplicitUsings>
    <EnableDefaultCompileItems>false</EnableDefaultCompileItems>
    <GenerateAssemblyInfo>false</GenerateAssemblyInfo>
    <TreatWarningsAsErrors>false</TreatWarningsAsErrors>
  </PropertyGroup>

  <ItemGroup>

    <Compile Include="Migration.cs" />

    <Compile Include="TestTypes.cs" />

    <FrameworkReference Include="Microsoft.AspNetCore.App" />

    <PackageReference
        Include="Microsoft.EntityFrameworkCore"
        Version="10.0.0" />

  </ItemGroup>

</Project>
"""

PROJECT_FILE.write_text(
    project_configuration.strip(),
    encoding="utf-8"
)


# ============================================================
# 4. Create Minimal EF Core Test Types
# ============================================================

# These classes provide the surrounding types required by
# our EF Core migration examples.
#
# They are test fixtures, not generated model output.

test_types_source = """
using Microsoft.EntityFrameworkCore;

public class StoreContext : DbContext
{
    public StoreContext(
        DbContextOptions<StoreContext> options
    ) : base(options)
    {
    }

    public DbSet<Order> Orders => Set<Order>();

    public DbSet<Product> Products => Set<Product>();

    public DbSet<Customer> Customers => Set<Customer>();
}

public class Order
{
    public int Id { get; set; }

    public bool IsOpen { get; set; }
}

public class Product
{
    public int Id { get; set; }
}

public class Customer
{
    public int Id { get; set; }

    public string Name { get; set; } = "";
}
"""

TEST_TYPES_FILE.write_text(
    test_types_source.strip(),
    encoding="utf-8"
)


# ============================================================
# 5. Write Initial Reference Code
# ============================================================

SOURCE_FILE.write_text(
    heldout_test_dataset[0]["modern_code"],
    encoding="utf-8"
)


# ============================================================
# 6. Restore Dependencies
# ============================================================

print("\n--- Restoring .NET Dependencies ---")

dotnet_environment = os.environ.copy()

dotnet_environment["DOTNET_CLI_TELEMETRY_OPTOUT"] = "1"
dotnet_environment["DOTNET_SKIP_FIRST_TIME_EXPERIENCE"] = "1"

restore_result = subprocess.run(
    [
        str(dotnet_path),
        "restore",
        str(PROJECT_FILE),
        "--nologo",
        "--verbosity",
        "quiet"
    ],
    cwd=str(COMPILE_DIR),
    env=dotnet_environment,
    capture_output=True,
    text=True,
    timeout=240
)

if restore_result.returncode != 0:

    print(restore_result.stdout)
    print(restore_result.stderr)

    raise RuntimeError(
        ".NET dependency restoration failed"
    )

print("Dependencies restored: PASS")


# ============================================================
# 7. Compilation Function
# ============================================================

def compile_csharp(source_code):

    SOURCE_FILE.write_text(
        source_code,
        encoding="utf-8"
    )

    result = subprocess.run(
        [
            str(dotnet_path),
            "build",
            str(PROJECT_FILE),
            "--no-restore",
            "--no-incremental",
            "--nologo",
            "--verbosity",
            "quiet"
        ],
        cwd=str(COMPILE_DIR),
        env=dotnet_environment,
        capture_output=True,
        text=True,
        timeout=120
    )

    compiler_output = (
        result.stdout + "\n" + result.stderr
    ).strip()

    return {
        "success": result.returncode == 0,
        "return_code": result.returncode,
        "compiler_output": compiler_output
    }


# ============================================================
# 8. Compile All Reference Modern C# Examples
# ============================================================

reference_compile_results = []

print("\n--- Reference Code Compilation ---")

for example in heldout_test_dataset:

    migration_id = example["migration_id"]

    compilation = compile_csharp(
        example["modern_code"]
    )

    reference_compile_results.append({
        "migration_id": migration_id,
        **compilation
    })

    status = (
        "PASS"
        if compilation["success"]
        else "FAIL"
    )

    print(f"{migration_id}: {status}")

    if not compilation["success"]:

        print("\nCompiler output:")
        print(compilation["compiler_output"])


# ============================================================
# 9. Summary
# ============================================================

successful = sum(
    result["success"]
    for result in reference_compile_results
)

total = len(reference_compile_results)

print("\n" + "=" * 60)
print("REFERENCE COMPILATION RESULTS")
print("=" * 60)

print("Total reference codes:", total)
print("Successfully compiled:", successful)
print("Failed:", total - successful)


if successful == total:

    print("\nReference compilation: PASS")

else:

    print(
        "\nReference compilation: NEEDS REVIEW"
    )

# Model Compilation Benchmark

Here, we will distinguish between two different success criteria:
- Format compliance: Did the model produce only C# code?
- Compilation success: Does the generated C# code actually compile?

In [ ]:
import re
import json
import pandas as pd

from pathlib import Path


# ============================================================
# 1. Verify Compilation Environment
# ============================================================

assert callable(compile_csharp)

assert all(
    result["success"]
    for result in reference_compile_results
)

print("Reference compilation: 10/10 PASS")


# ============================================================
# 2. Load Model Outputs
# ============================================================

generation_file = (
    Path(EXPERIMENT_DIR)
    / "evaluation"
    / "generation_results_v1.jsonl"
)

with open(
    generation_file,
    "r",
    encoding="utf-8"
) as file:

    generated_results = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

assert len(generated_results) == 30


# ============================================================
# 3. Extract C# Code Without Modifying It
# ============================================================

def extract_csharp_code(raw_output):

    text = raw_output.strip()

    # Case 1: Markdown code block.
    fenced_match = re.search(
        r"```[^\n]*\n(.*?)(?:\n```|\Z)",
        text,
        flags=re.DOTALL
    )

    if fenced_match:

        return (
            fenced_match.group(1).strip(),
            "markdown_fence"
        )

    # Case 2: Explanatory text preceding C# code.
    csharp_start = re.search(
        r"(?m)^[ \t]*(?:"
        r"using\s+[A-Za-z_][\w.]*\s*;"
        r"|namespace\s+[A-Za-z_]"
        r"|(?:public|internal)\s+"
        r"(?:(?:sealed|static|abstract|partial)\s+)*"
        r"(?:class|interface|record|struct)\b"
        r")",
        text
    )

    if csharp_start:

        extracted = text[
            csharp_start.start():
        ].strip()

        return (
            extracted,
            "csharp_start"
        )

    # Case 3: No recognizable structure.
    return (
        text,
        "unchanged"
    )


# ============================================================
# 4. Compilation Result Storage
# ============================================================

compilation_file = (
    Path(EXPERIMENT_DIR)
    / "evaluation"
    / "model_compilation_v1.jsonl"
)

compilation_results = []

if compilation_file.exists():

    with open(
        compilation_file,
        "r",
        encoding="utf-8"
    ) as file:

        compilation_results = [
            json.loads(line)
            for line in file
            if line.strip()
        ]


completed_keys = {
    (
        result["model"],
        result["migration_id"]
    )
    for result in compilation_results
}

assert len(completed_keys) == len(compilation_results)

print(
    "\nPreviously compiled outputs:",
    len(compilation_results)
)


# ============================================================
# 5. Compile All Generated C# Codes
# ============================================================

print("\n" + "=" * 65)
print("MODEL COMPILATION BENCHMARK")
print("=" * 65)

for result in generated_results:

    model_name = result["model"]
    migration_id = result["migration_id"]

    key = (
        model_name,
        migration_id
    )

    if key in completed_keys:
        continue

    generated_code = result["generated_code"]

    # Extract code, without repairing it.
    extracted_code, extraction_method = (
        extract_csharp_code(generated_code)
    )

    # Compile using the verified .NET test project.
    compilation = compile_csharp(
        extracted_code
    )

    compiler_output = compilation["compiler_output"]

    # Collect C# compiler diagnostics.
    diagnostic_codes = sorted(
        set(
            re.findall(
                r"\b(?:error|warning)\s+(?:CS|NU)\d+",
                compiler_output,
                flags=re.IGNORECASE
            )
        )
    )

    compilation_record = {
        "model": model_name,

        "migration_id": migration_id,

        "group_id": result["group_id"],

        "compile_success": compilation["success"],

        "return_code": compilation["return_code"],

        "extraction_method": extraction_method,

        "diagnostic_codes": diagnostic_codes,

        "compiler_output": compiler_output,

        "extracted_code": extracted_code
    }

    # Save each result immediately.
    with open(
        compilation_file,
        "a",
        encoding="utf-8"
    ) as file:

        file.write(
            json.dumps(
                compilation_record,
                ensure_ascii=False
            ) + "\n"
        )

    compilation_results.append(
        compilation_record
    )

    completed_keys.add(key)

    status = (
        "PASS"
        if compilation["success"]
        else "FAIL"
    )

    print(
        f"{model_name:16s} | "
        f"{migration_id:25s} | "
        f"{status}"
    )

    if not compilation["success"]:

        error_lines = [
            line.strip()
            for line in compiler_output.splitlines()
            if re.search(
                r"\berror\s+(?:CS|NU)\d+",
                line,
                flags=re.IGNORECASE
            )
        ]

        for error_line in error_lines[:3]:
            print("   ", error_line[:250])


# ============================================================
# 6. Verify All Results
# ============================================================

expected_keys = {
    (
        result["model"],
        result["migration_id"]
    )
    for result in generated_results
}

assert completed_keys == expected_keys

assert len(compilation_results) == 30


# ============================================================
# 7. Create Evaluation DataFrame
# ============================================================

compilation_df = pd.DataFrame(
    compilation_results
)

model_order = [
    "base_codellama",
    "optuna_best",
    "mc_dropout"
]

compilation_df["model"] = pd.Categorical(
    compilation_df["model"],
    categories=model_order,
    ordered=True
)


# ============================================================
# 8. Aggregate Compilation Metrics
# ============================================================

compilation_summary = compilation_df.groupby(
    "model",
    observed=True
).agg(

    total_examples=(
        "migration_id",
        "count"
    ),

    compiled=(
        "compile_success",
        "sum"
    ),

    compilation_rate=(
        "compile_success",
        "mean"
    )

)


# ============================================================
# 9. Display Results
# ============================================================

print("\n" + "=" * 65)
print("COMPILATION RESULTS")
print("=" * 65)

print(
    compilation_summary.round(4).to_string()
)


# ============================================================
# 10. Compare Compilation by Migration Family
# ============================================================

family_results = compilation_df.groupby(
    ["model", "group_id"],
    observed=True
)["compile_success"].agg(
    ["sum", "count", "mean"]
)

print("\n--- Compilation by Migration Family ---")

print(
    family_results.round(4).to_string()
)


# ============================================================
# 11. Save Summary
# ============================================================

summary_path = (
    Path(EXPERIMENT_DIR)
    / "evaluation"
    / "compilation_summary_v1.csv"
)

compilation_summary.to_csv(
    summary_path
)

print("\nSaved results:")
print(compilation_file)

print("\nSaved summary:")
print(summary_path)

print("\nModel compilation benchmark: PASS")

# Behavioral Testing: Configuration Migration

In [ ]:
import json
import subprocess
import pandas as pd

from pathlib import Path


# ============================================================
# 1. Test Project Directory
# ============================================================

BEHAVIOR_DIR = Path(
    "/content/codemigrate_behavior_tests/config"
)

BEHAVIOR_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEHAVIOR_PROJECT = (
    BEHAVIOR_DIR / "BehaviorConfig.csproj"
)

BEHAVIOR_SOURCE = (
    BEHAVIOR_DIR / "Migration.cs"
)

BEHAVIOR_RUNNER = (
    BEHAVIOR_DIR / "Program.cs"
)


# ============================================================
# 2. Create .NET Console Project
# ============================================================

BEHAVIOR_PROJECT.write_text(
    """
<Project Sdk="Microsoft.NET.Sdk">

  <PropertyGroup>
    <OutputType>Exe</OutputType>
    <TargetFramework>net10.0</TargetFramework>
    <Nullable>enable</Nullable>
    <ImplicitUsings>disable</ImplicitUsings>
    <EnableDefaultCompileItems>false</EnableDefaultCompileItems>
  </PropertyGroup>

  <ItemGroup>
    <Compile Include="Migration.cs" />
    <Compile Include="Program.cs" />
    <FrameworkReference Include="Microsoft.AspNetCore.App" />
  </ItemGroup>

</Project>
""".strip(),
    encoding="utf-8"
)


# ============================================================
# 3. C# Runtime Test Program
# ============================================================

runner_source = r"""
using System;
using System.Collections.Generic;
using System.Reflection;
using Microsoft.Extensions.Configuration;

internal static class Program
{
    private static IConfiguration BuildConfig(
        Dictionary<string, string?> values)
    {
        return new ConfigurationBuilder()
            .AddInMemoryCollection(values)
            .Build();
    }

    private static string InvokeMethod(
        string className,
        string methodName,
        IConfiguration configuration)
    {
        Type type =
            Assembly.GetExecutingAssembly().GetType(className)
            ?? throw new Exception(
                "Class not found: " + className);

        ConstructorInfo constructor =
            type.GetConstructor(new[] { typeof(IConfiguration) })
            ?? throw new Exception(
                "IConfiguration constructor not found");

        object instance = constructor.Invoke(
            new object[] { configuration });

        MethodInfo method =
            type.GetMethod(methodName)
            ?? throw new Exception(
                "Method not found: " + methodName);

        object? result = method.Invoke(
            instance,
            null);

        return result as string
            ?? throw new Exception(
                "Method returned null or non-string value");
    }

    private static void AssertEqual(
        string actual,
        string expected,
        string testName)
    {
        if (actual != expected)
        {
            throw new Exception(
                testName +
                " | Expected: " + expected +
                " | Actual: " + actual);
        }
    }

    private static void TestConfigDefault()
    {
        var withRegion = BuildConfig(
            new Dictionary<string, string?>
            {
                ["Region"] = "Europe"
            });

        string actual = InvokeMethod(
            "RegionSettings",
            "GetRegion",
            withRegion);

        AssertEqual(
            actual,
            "Europe",
            "Existing region");

        var withoutRegion = BuildConfig(
            new Dictionary<string, string?>());

        actual = InvokeMethod(
            "RegionSettings",
            "GetRegion",
            withoutRegion);

        AssertEqual(
            actual,
            "global",
            "Default region");
    }

    private static void TestConnectionString()
    {
        var configuration = BuildConfig(
            new Dictionary<string, string?>
            {
                ["ConnectionStrings:MainDatabase"] =
                    "Server=correct;Database=Catalog;",

                // Deliberate trap:
                // This is NOT the expected configuration path.
                ["MainDatabase:ConnectionString"] =
                    "Server=wrong;Database=Wrong;"
            });

        string actual = InvokeMethod(
            "DatabaseSettings",
            "GetConnection",
            configuration);

        AssertEqual(
            actual,
            "Server=correct;Database=Catalog;",
            "Connection string lookup");

        var missingConfiguration = BuildConfig(
            new Dictionary<string, string?>());

        actual = InvokeMethod(
            "DatabaseSettings",
            "GetConnection",
            missingConfiguration);

        AssertEqual(
            actual,
            "",
            "Missing connection fallback");
    }

    private static int Main(string[] args)
    {
        try
        {
            if (args.Length != 1)
                throw new Exception("Migration ID required");

            switch (args[0])
            {
                case "test_config_default":
                    TestConfigDefault();
                    break;

                case "test_connection_string":
                    TestConnectionString();
                    break;

                default:
                    throw new Exception("Unknown migration ID");
            }

            Console.WriteLine("BEHAVIOR PASS");
            return 0;
        }
        catch (Exception exception)
        {
            Console.WriteLine(
                "BEHAVIOR FAIL: " + exception.Message);

            return 1;
        }
    }
}
"""

BEHAVIOR_RUNNER.write_text(
    runner_source.strip(),
    encoding="utf-8"
)


# ============================================================
# 4. Load Model Compilation Results
# ============================================================

compilation_file = (
    Path(EXPERIMENT_DIR)
    / "evaluation"
    / "model_compilation_v1.jsonl"
)

with open(
    compilation_file,
    "r",
    encoding="utf-8"
) as file:

    compiled_outputs = [
        json.loads(line)
        for line in file
        if line.strip()
    ]


# ============================================================
# 5. Prepare .NET Project
# ============================================================

test_ids = [
    "test_config_default",
    "test_connection_string"
]

test_lookup = {
    example["migration_id"]: example
    for example in heldout_test_dataset
}

# Write reference source before restoring.
BEHAVIOR_SOURCE.write_text(
    test_lookup["test_config_default"]["modern_code"],
    encoding="utf-8"
)

restore = subprocess.run(
    [
        str(dotnet_path),
        "restore",
        str(BEHAVIOR_PROJECT),
        "--nologo",
        "--verbosity",
        "quiet"
    ],
    cwd=str(BEHAVIOR_DIR),
    env=dotnet_environment,
    capture_output=True,
    text=True,
    timeout=180
)

assert restore.returncode == 0, (
    restore.stdout + "\n" + restore.stderr
)


# ============================================================
# 6. Compile and Execute a Behavioral Test
# ============================================================

def run_behavior_test(migration_id, source_code):

    BEHAVIOR_SOURCE.write_text(
        source_code,
        encoding="utf-8"
    )

    build = subprocess.run(
        [
            str(dotnet_path),
            "build",
            str(BEHAVIOR_PROJECT),
            "--no-restore",
            "--no-incremental",
            "--nologo",
            "--verbosity",
            "quiet"
        ],
        cwd=str(BEHAVIOR_DIR),
        env=dotnet_environment,
        capture_output=True,
        text=True,
        timeout=120
    )

    if build.returncode != 0:

        return {
            "passed": False,
            "stage": "compilation",
            "details": (
                build.stdout + "\n" + build.stderr
            ).strip()
        }

    assembly = (
        BEHAVIOR_DIR
        / "bin"
        / "Debug"
        / "net10.0"
        / "BehaviorConfig.dll"
    )

    execution = subprocess.run(
        [
            str(dotnet_path),
            str(assembly),
            migration_id
        ],
        cwd=str(BEHAVIOR_DIR),
        env=dotnet_environment,
        capture_output=True,
        text=True,
        timeout=60
    )

    return {
        "passed": execution.returncode == 0,
        "stage": "runtime",
        "details": (
            execution.stdout + "\n" + execution.stderr
        ).strip()
    }


# ============================================================
# 7. Validate Reference Implementations First
# ============================================================

print("--- Reference Behavioral Tests ---")

for migration_id in test_ids:

    reference_code = test_lookup[
        migration_id
    ]["modern_code"]

    outcome = run_behavior_test(
        migration_id,
        reference_code
    )

    print(
        migration_id,
        "PASS" if outcome["passed"] else "FAIL"
    )

    if not outcome["passed"]:
        print(outcome["details"])

    assert outcome["passed"], (
        "Reference behavior test failed"
    )


# ============================================================
# 8. Evaluate All Three Models
# ============================================================

behavior_results = []

print("\n--- Model Behavioral Tests ---")

for model_name in [
    "base_codellama",
    "optuna_best",
    "mc_dropout"
]:

    for migration_id in test_ids:

        compilation_record = next(
            result
            for result in compiled_outputs
            if result["model"] == model_name
            and result["migration_id"] == migration_id
        )

        outcome = run_behavior_test(
            migration_id,
            compilation_record["extracted_code"]
        )

        behavior_results.append({
            "model": model_name,
            "migration_id": migration_id,
            **outcome
        })

        print(
            f"{model_name:16s} | "
            f"{migration_id:25s} | "
            f"{'PASS' if outcome['passed'] else 'FAIL'}"
        )

        if not outcome["passed"]:

            detail_lines = [
                line.strip()
                for line in outcome["details"].splitlines()
                if line.strip()
            ]

            print(
                "  Reason:",
                detail_lines[-1][:250]
                if detail_lines
                else "Unknown error"
            )


# ============================================================
# 9. Summarize Results
# ============================================================

behavior_df = pd.DataFrame(
    behavior_results
)

print("\n--- Configuration Behavior Summary ---")

print(
    behavior_df.groupby("model")["passed"]
    .agg(["sum", "count", "mean"])
    .to_string()
)


# ============================================================
# 10. Save Results
# ============================================================

output_path = (
    Path(EXPERIMENT_DIR)
    / "evaluation"
    / "behavior_configuration_v1.csv"
)

behavior_df.to_csv(
    output_path,
    index=False
)

print("\nSaved:", output_path)

print("\nConfiguration behavioral evaluation: PASS")

# HTTP Context & MVC Behavioral Testing

In [ ]:
import json
import subprocess
import pandas as pd

from pathlib import Path


# ============================================================
# 1. Create HTTP Behavioral Test Project
# ============================================================

HTTP_TEST_DIR = Path(
    "/content/codemigrate_behavior_tests/http_mvc"
)

HTTP_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HTTP_PROJECT = HTTP_TEST_DIR / "BehaviorHttp.csproj"
HTTP_SOURCE = HTTP_TEST_DIR / "Migration.cs"
HTTP_RUNNER = HTTP_TEST_DIR / "Program.cs"


HTTP_PROJECT.write_text(
    """
<Project Sdk="Microsoft.NET.Sdk">

  <PropertyGroup>
    <OutputType>Exe</OutputType>
    <TargetFramework>net10.0</TargetFramework>
    <Nullable>enable</Nullable>
    <ImplicitUsings>disable</ImplicitUsings>
    <EnableDefaultCompileItems>false</EnableDefaultCompileItems>
  </PropertyGroup>

  <ItemGroup>
    <Compile Include="Migration.cs" />
    <Compile Include="Program.cs" />
    <FrameworkReference Include="Microsoft.AspNetCore.App" />
  </ItemGroup>

</Project>
""".strip(),
    encoding="utf-8"
)


# ============================================================
# 2. Create C# Behavioral Test Runner
# ============================================================

http_runner_source = r"""
using System;
using System.Net;
using System.Reflection;
using Microsoft.AspNetCore.Http;
using Microsoft.AspNetCore.Mvc;

internal static class Program
{
    private static (ControllerBase, DefaultHttpContext)
        CreateController(string className)
    {
        Type type =
            Assembly.GetExecutingAssembly().GetType(className)
            ?? throw new Exception(
                "Class not found: " + className);

        object instance =
            Activator.CreateInstance(type)
            ?? throw new Exception(
                "Cannot create controller: " + className);

        if (instance is not ControllerBase controller)
        {
            throw new Exception(
                "Class is not an ASP.NET Core ControllerBase");
        }

        var context = new DefaultHttpContext();

        controller.ControllerContext = new ControllerContext
        {
            HttpContext = context
        };

        return (controller, context);
    }

    private static object? Invoke(
        ControllerBase controller,
        string methodName,
        params object[] arguments)
    {
        MethodInfo method =
            controller.GetType().GetMethod(methodName)
            ?? throw new Exception(
                "Method not found: " + methodName);

        return method.Invoke(
            controller,
            arguments);
    }

    private static void Assert(
        bool condition,
        string description)
    {
        if (!condition)
        {
            throw new Exception(description);
        }
    }

    private static int? GetStatusCode(object? result)
    {
        return result switch
        {
            StatusCodeResult status => status.StatusCode,
            ObjectResult obj => obj.StatusCode,
            _ => null
        };
    }

    private static void TestHttps()
    {
        var (controller, context) =
            CreateController("SecurityController");

        context.Request.Scheme = "https";

        Assert(
            Invoke(controller, "IsSecure") is bool secure
                && secure,
            "HTTPS request must return true");

        context.Request.Scheme = "http";

        Assert(
            Invoke(controller, "IsSecure") is bool insecure
                && !insecure,
            "HTTP request must return false");
    }

    private static void TestHttpMethod()
    {
        var (controller, context) =
            CreateController("RequestInfoController");

        context.Request.Method = "PATCH";

        Assert(
            Invoke(controller, "GetMethod") is string method
                && method == "PATCH",
            "PATCH method was not preserved");

        context.Request.Method = "DELETE";

        Assert(
            Invoke(controller, "GetMethod") is string method2
                && method2 == "DELETE",
            "DELETE method was not preserved");
    }

    private static void TestClientIp()
    {
        var (controller, context) =
            CreateController("NetworkController");

        context.Connection.RemoteIpAddress =
            IPAddress.Parse("203.0.113.10");

        Assert(
            Invoke(controller, "GetClientIp") is string ipv4
                && ipv4 == "203.0.113.10",
            "IPv4 address was not preserved");

        var ipv6Address = IPAddress.Parse("2001:db8::5");

        context.Connection.RemoteIpAddress = ipv6Address;

        Assert(
            Invoke(controller, "GetClientIp") is string ipv6
                && ipv6 == ipv6Address.ToString(),
            "IPv6 address was not preserved");

        context.Connection.RemoteIpAddress = null;

        Assert(
            Invoke(controller, "GetClientIp") is null,
            "Null IP behavior was not preserved");
    }

    private static void TestHttpNotFound()
    {
        var (controller, _) =
            CreateController("ProductController");

        object? invalidResult = Invoke(
            controller,
            "Details",
            0);

        Assert(
            GetStatusCode(invalidResult) == 404,
            "Invalid ID must return HTTP 404");

        object? validResult = Invoke(
            controller,
            "Details",
            42);

        Assert(
            validResult is ContentResult content
                && content.Content == "42",
            "Valid ID must return the original content");
    }

    private static void TestHttpStatus()
    {
        var (controller, _) =
            CreateController("AccessController");

        object? result = Invoke(
            controller,
            "Denied");

        Assert(
            GetStatusCode(result) == 403,
            "Denied must return HTTP 403");
    }

    private static int Main(string[] args)
    {
        try
        {
            if (args.Length != 1)
            {
                throw new Exception(
                    "Migration ID is required");
            }

            switch (args[0])
            {
                case "test_https":
                    TestHttps();
                    break;

                case "test_http_method":
                    TestHttpMethod();
                    break;

                case "test_client_ip":
                    TestClientIp();
                    break;

                case "test_http_not_found":
                    TestHttpNotFound();
                    break;

                case "test_http_status":
                    TestHttpStatus();
                    break;

                default:
                    throw new Exception(
                        "Unknown migration: " + args[0]);
            }

            Console.WriteLine("BEHAVIOR PASS");

            return 0;
        }
        catch (Exception exception)
        {
            Console.WriteLine(
                "BEHAVIOR FAIL: " + exception.Message);

            return 1;
        }
    }
}
"""

HTTP_RUNNER.write_text(
    http_runner_source.strip(),
    encoding="utf-8"
)


# ============================================================
# 3. Select Test Cases
# ============================================================

HTTP_TEST_IDS = [
    "test_https",
    "test_http_method",
    "test_client_ip",
    "test_http_not_found",
    "test_http_status"
]

test_lookup = {
    example["migration_id"]: example
    for example in heldout_test_dataset
}


# ============================================================
# 4. Restore Project
# ============================================================

HTTP_SOURCE.write_text(
    test_lookup["test_https"]["modern_code"],
    encoding="utf-8"
)

restore = subprocess.run(
    [
        str(dotnet_path),
        "restore",
        str(HTTP_PROJECT),
        "--nologo",
        "--verbosity",
        "quiet"
    ],
    cwd=str(HTTP_TEST_DIR),
    env=dotnet_environment,
    capture_output=True,
    text=True,
    timeout=180
)

assert restore.returncode == 0, (
    restore.stdout + "\n" + restore.stderr
)


# ============================================================
# 5. Define Compile + Runtime Test
# ============================================================

def run_http_behavior_test(migration_id, source_code):

    HTTP_SOURCE.write_text(
        source_code,
        encoding="utf-8"
    )

    build = subprocess.run(
        [
            str(dotnet_path),
            "build",
            str(HTTP_PROJECT),
            "--no-restore",
            "--no-incremental",
            "--nologo",
            "--verbosity",
            "quiet"
        ],
        cwd=str(HTTP_TEST_DIR),
        env=dotnet_environment,
        capture_output=True,
        text=True,
        timeout=120
    )

    if build.returncode != 0:

        return {
            "passed": False,
            "stage": "compilation",
            "details": (
                build.stdout + "\n" + build.stderr
            ).strip()
        }

    assembly = (
        HTTP_TEST_DIR
        / "bin"
        / "Debug"
        / "net10.0"
        / "BehaviorHttp.dll"
    )

    execution = subprocess.run(
        [
            str(dotnet_path),
            str(assembly),
            migration_id
        ],
        cwd=str(HTTP_TEST_DIR),
        env=dotnet_environment,
        capture_output=True,
        text=True,
        timeout=60
    )

    return {
        "passed": execution.returncode == 0,
        "stage": "runtime",
        "details": (
            execution.stdout + "\n" + execution.stderr
        ).strip()
    }


# ============================================================
# 6. Validate Reference Implementations
# ============================================================

print("--- Reference HTTP/MVC Behavioral Tests ---")

for migration_id in HTTP_TEST_IDS:

    reference_code = test_lookup[
        migration_id
    ]["modern_code"]

    outcome = run_http_behavior_test(
        migration_id,
        reference_code
    )

    print(
        migration_id,
        "PASS" if outcome["passed"] else "FAIL"
    )

    if not outcome["passed"]:
        print(outcome["details"])

    assert outcome["passed"], (
        "Reference test failed: " + migration_id
    )


# ============================================================
# 7. Load Compiled Model Outputs
# ============================================================

compilation_file = (
    Path(EXPERIMENT_DIR)
    / "evaluation"
    / "model_compilation_v1.jsonl"
)

with open(
    compilation_file,
    "r",
    encoding="utf-8"
) as file:

    compiled_outputs = [
        json.loads(line)
        for line in file
        if line.strip()
    ]


# ============================================================
# 8. Evaluate Model Outputs
# ============================================================

http_behavior_results = []

print("\n--- Model HTTP/MVC Behavioral Tests ---")

for model_name in [
    "base_codellama",
    "optuna_best",
    "mc_dropout"
]:

    for migration_id in HTTP_TEST_IDS:

        record = next(
            result
            for result in compiled_outputs
            if result["model"] == model_name
            and result["migration_id"] == migration_id
        )

        if not record["compile_success"]:

            outcome = {
                "passed": False,
                "stage": "compilation",
                "details": "Failed original compilation test"
            }

        else:

            outcome = run_http_behavior_test(
                migration_id,
                record["extracted_code"]
            )

        http_behavior_results.append({
            "model": model_name,
            "migration_id": migration_id,
            **outcome
        })

        status = (
            "PASS"
            if outcome["passed"]
            else "FAIL"
        )

        print(
            f"{model_name:16s} | "
            f"{migration_id:22s} | "
            f"{status}"
        )

        if not outcome["passed"]:

            detail_lines = [
                line.strip()
                for line in outcome["details"].splitlines()
                if line.strip()
            ]

            print(
                "  Reason:",
                detail_lines[-1][:200]
                if detail_lines
                else "Unknown error"
            )


# ============================================================
# 9. Summarize Results
# ============================================================

http_behavior_df = pd.DataFrame(
    http_behavior_results
)

print("\n" + "=" * 60)
print("HTTP/MVC BEHAVIOR RESULTS")
print("=" * 60)

summary = http_behavior_df.groupby(
    "model"
)["passed"].agg(
    ["sum", "count", "mean"]
)

print(summary.to_string())


# ============================================================
# 10. Save Results
# ============================================================

output_path = (
    Path(EXPERIMENT_DIR)
    / "evaluation"
    / "behavior_http_mvc_v1.csv"
)

http_behavior_df.to_csv(
    output_path,
    index=False
)

print("\nSaved:", output_path)

print("\nHTTP/MVC behavioral evaluation: PASS")

# Entity Framework Behavioral Testing

In [ ]:
import json
import subprocess
import pandas as pd

from pathlib import Path


# ============================================================
# 1. Create EF Core Test Project
# ============================================================

EF_TEST_DIR = Path(
    "/content/codemigrate_behavior_tests/ef_core"
)

EF_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EF_PROJECT = EF_TEST_DIR / "BehaviorEF.csproj"
EF_SOURCE = EF_TEST_DIR / "Migration.cs"
EF_RUNNER = EF_TEST_DIR / "Program.cs"
EF_TEST_TYPES = EF_TEST_DIR / "TestTypes.cs"


# ============================================================
# 2. Create .NET Project
# ============================================================

EF_PROJECT.write_text(
    """
<Project Sdk="Microsoft.NET.Sdk">

  <PropertyGroup>
    <OutputType>Exe</OutputType>
    <TargetFramework>net10.0</TargetFramework>
    <Nullable>enable</Nullable>
    <ImplicitUsings>disable</ImplicitUsings>
    <EnableDefaultCompileItems>false</EnableDefaultCompileItems>
  </PropertyGroup>

  <ItemGroup>

    <Compile Include="Migration.cs" />
    <Compile Include="TestTypes.cs" />
    <Compile Include="Program.cs" />

    <PackageReference
        Include="Microsoft.EntityFrameworkCore.Sqlite"
        Version="10.0.0" />

  </ItemGroup>

</Project>
""".strip(),
    encoding="utf-8"
)


# ============================================================
# 3. Reuse Previously Validated EF Types
# ============================================================

original_test_types = Path(
    "/content/codemigrate_compile_tests/TestTypes.cs"
)

assert original_test_types.exists()

EF_TEST_TYPES.write_text(
    original_test_types.read_text(encoding="utf-8"),
    encoding="utf-8"
)


# ============================================================
# 4. Create C# Runtime Test Runner
# ============================================================

ef_runner_source = r"""
using System;
using System.Collections.Generic;
using System.Linq;
using System.Reflection;
using System.Threading.Tasks;
using Microsoft.Data.Sqlite;
using Microsoft.EntityFrameworkCore;

internal static class Program
{
    private static async Task<T> InvokeAsync<T>(
        string className,
        string methodName,
        StoreContext db,
        params object[] arguments)
    {
        Type type =
            Assembly.GetExecutingAssembly().GetType(className)
            ?? throw new Exception(
                "Class not found: " + className);

        ConstructorInfo constructor =
            type.GetConstructor(new[] { typeof(StoreContext) })
            ?? throw new Exception(
                "StoreContext constructor not found");

        object instance = constructor.Invoke(
            new object[] { db });

        MethodInfo method =
            type.GetMethod(methodName)
            ?? throw new Exception(
                "Method not found: " + methodName);

        object? result = method.Invoke(
            instance,
            arguments);

        if (result is not Task<T> task)
        {
            throw new Exception(
                "Expected Task<T> return type");
        }

        return await task;
    }

    private static void Assert(
        bool condition,
        string description)
    {
        if (!condition)
        {
            throw new Exception(description);
        }
    }

    private static StoreContext CreateContext(
        SqliteConnection connection)
    {
        var options =
            new DbContextOptionsBuilder<StoreContext>()
                .UseSqlite(connection)
                .Options;

        var db = new StoreContext(options);

        db.Database.EnsureCreated();

        return db;
    }

    private static async Task TestAnyAsync()
    {
        using var connection =
            new SqliteConnection("Data Source=:memory:");

        connection.Open();

        using var db = CreateContext(connection);

        db.Orders.AddRange(
            new Order { Id = 1, IsOpen = false },
            new Order { Id = 2, IsOpen = true }
        );

        db.SaveChanges();

        bool hasOpenOrders = await InvokeAsync<bool>(
            "OrderQueries",
            "HasOpenOrders",
            db);

        Assert(
            hasOpenOrders,
            "Open order must be detected");

        var openOrder = db.Orders.Single(
            order => order.Id == 2);

        db.Orders.Remove(openOrder);

        db.SaveChanges();

        hasOpenOrders = await InvokeAsync<bool>(
            "OrderQueries",
            "HasOpenOrders",
            db);

        Assert(
            !hasOpenOrders,
            "No open orders should return false");
    }

    private static async Task TestFirstOrDefaultAsync()
    {
        using var connection =
            new SqliteConnection("Data Source=:memory:");

        connection.Open();

        using var db = CreateContext(connection);

        db.Products.AddRange(
            new Product { Id = 10 },
            new Product { Id = 20 }
        );

        db.SaveChanges();

        Product? found = await InvokeAsync<Product?>(
            "ProductQueries",
            "GetById",
            db,
            20);

        Assert(
            found != null && found.Id == 20,
            "Product lookup returned wrong item");

        Product? missing = await InvokeAsync<Product?>(
            "ProductQueries",
            "GetById",
            db,
            999);

        Assert(
            missing == null,
            "Missing product must return null");
    }

    private static async Task TestToListAsync()
    {
        using var connection =
            new SqliteConnection("Data Source=:memory:");

        connection.Open();

        using var db = CreateContext(connection);

        List<Customer> empty =
            await InvokeAsync<List<Customer>>(
                "CustomerQueries",
                "GetCustomers",
                db);

        Assert(
            empty.Count == 0,
            "Empty database must return empty list");

        db.Customers.AddRange(
            new Customer { Id = 1, Name = "Zulu" },
            new Customer { Id = 2, Name = "Alpha" },
            new Customer { Id = 3, Name = "Mike" }
        );

        db.SaveChanges();

        List<Customer> customers =
            await InvokeAsync<List<Customer>>(
                "CustomerQueries",
                "GetCustomers",
                db);

        string[] actualNames = customers
            .Select(customer => customer.Name)
            .ToArray();

        string[] expectedNames = {
            "Alpha", "Mike", "Zulu"
        };

        Assert(
            actualNames.SequenceEqual(expectedNames),
            "Customer ordering was not preserved");
    }

    private static async Task<int> Main(string[] args)
    {
        try
        {
            if (args.Length != 1)
            {
                throw new Exception(
                    "Migration ID required");
            }

            switch (args[0])
            {
                case "test_ef_any":
                    await TestAnyAsync();
                    break;

                case "test_ef_first":
                    await TestFirstOrDefaultAsync();
                    break;

                case "test_ef_list":
                    await TestToListAsync();
                    break;

                default:
                    throw new Exception(
                        "Unknown migration ID");
            }

            Console.WriteLine("BEHAVIOR PASS");

            return 0;
        }
        catch (Exception exception)
        {
            Console.WriteLine(
                "BEHAVIOR FAIL: " + exception.Message);

            return 1;
        }
    }
}
"""

EF_RUNNER.write_text(
    ef_runner_source.strip(),
    encoding="utf-8"
)


# ============================================================
# 5. Test Cases
# ============================================================

EF_TEST_IDS = [
    "test_ef_any",
    "test_ef_first",
    "test_ef_list"
]

test_lookup = {
    example["migration_id"]: example
    for example in heldout_test_dataset
}


# ============================================================
# 6. Restore Dependencies
# ============================================================

EF_SOURCE.write_text(
    test_lookup["test_ef_any"]["modern_code"],
    encoding="utf-8"
)

restore = subprocess.run(
    [
        str(dotnet_path),
        "restore",
        str(EF_PROJECT),
        "--nologo",
        "--verbosity",
        "quiet"
    ],
    cwd=str(EF_TEST_DIR),
    env=dotnet_environment,
    capture_output=True,
    text=True,
    timeout=240
)

assert restore.returncode == 0, (
    restore.stdout + "\n" + restore.stderr
)

print("EF Core SQLite dependencies: PASS")


# ============================================================
# 7. Compile and Run Function
# ============================================================

def run_ef_behavior_test(migration_id, source_code):

    EF_SOURCE.write_text(
        source_code,
        encoding="utf-8"
    )

    build = subprocess.run(
        [
            str(dotnet_path),
            "build",
            str(EF_PROJECT),
            "--no-restore",
            "--no-incremental",
            "--nologo",
            "--verbosity",
            "quiet"
        ],
        cwd=str(EF_TEST_DIR),
        env=dotnet_environment,
        capture_output=True,
        text=True,
        timeout=120
    )

    if build.returncode != 0:

        return {
            "passed": False,
            "stage": "compilation",
            "details": (
                build.stdout + "\n" + build.stderr
            ).strip()
        }

    assembly = (
        EF_TEST_DIR
        / "bin"
        / "Debug"
        / "net10.0"
        / "BehaviorEF.dll"
    )

    execution = subprocess.run(
        [
            str(dotnet_path),
            str(assembly),
            migration_id
        ],
        cwd=str(EF_TEST_DIR),
        env=dotnet_environment,
        capture_output=True,
        text=True,
        timeout=60
    )

    return {
        "passed": execution.returncode == 0,
        "stage": "runtime",
        "details": (
            execution.stdout + "\n" + execution.stderr
        ).strip()
    }


# ============================================================
# 8. Verify Reference Implementations
# ============================================================

print("\n--- Reference EF Behavioral Tests ---")

for migration_id in EF_TEST_IDS:

    reference_code = test_lookup[
        migration_id
    ]["modern_code"]

    outcome = run_ef_behavior_test(
        migration_id,
        reference_code
    )

    print(
        migration_id,
        "PASS" if outcome["passed"] else "FAIL"
    )

    if not outcome["passed"]:
        print(outcome["details"])

    assert outcome["passed"], (
        "Reference EF behavior test failed"
    )


# ============================================================
# 9. Load Model Compilation Results
# ============================================================

compilation_file = (
    Path(EXPERIMENT_DIR)
    / "evaluation"
    / "model_compilation_v1.jsonl"
)

with open(
    compilation_file,
    "r",
    encoding="utf-8"
) as file:

    compiled_outputs = [
        json.loads(line)
        for line in file
        if line.strip()
    ]


# ============================================================
# 10. Evaluate Model Outputs
# ============================================================

ef_behavior_results = []

print("\n--- Model EF Behavioral Tests ---")

for model_name in [
    "base_codellama",
    "optuna_best",
    "mc_dropout"
]:

    for migration_id in EF_TEST_IDS:

        record = next(
            result
            for result in compiled_outputs
            if result["model"] == model_name
            and result["migration_id"] == migration_id
        )

        if not record["compile_success"]:

            outcome = {
                "passed": False,
                "stage": "compilation",
                "details": "Failed original compilation test"
            }

        else:

            outcome = run_ef_behavior_test(
                migration_id,
                record["extracted_code"]
            )

        ef_behavior_results.append({
            "model": model_name,
            "migration_id": migration_id,
            **outcome
        })

        print(
            f"{model_name:16s} | "
            f"{migration_id:20s} | "
            f"{'PASS' if outcome['passed'] else 'FAIL'}"
        )

        if not outcome["passed"]:

            detail_lines = [
                line.strip()
                for line in outcome["details"].splitlines()
                if line.strip()
            ]

            print(
                "  Reason:",
                detail_lines[-1][:200]
                if detail_lines
                else "Unknown error"
            )


# ============================================================
# 11. Summarize Results
# ============================================================

ef_behavior_df = pd.DataFrame(
    ef_behavior_results
)

print("\n" + "=" * 60)
print("ENTITY FRAMEWORK BEHAVIOR RESULTS")
print("=" * 60)

summary = ef_behavior_df.groupby(
    "model"
)["passed"].agg(
    ["sum", "count", "mean"]
)

print(summary.to_string())


# ============================================================
# 12. Save Results
# ============================================================

output_path = (
    Path(EXPERIMENT_DIR)
    / "evaluation"
    / "behavior_ef_v1.csv"
)

ef_behavior_df.to_csv(
    output_path,
    index=False
)

print("\nSaved:", output_path)

print("\nEF behavioral evaluation: PASS")

# Final Behavioral Comparison & Bootstrap

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path


# ============================================================
# 1. Load Behavioral Evaluation Results
# ============================================================

EVALUATION_DIR = (
    Path(EXPERIMENT_DIR)
    / "evaluation"
)

result_files = [
    "behavior_configuration_v1.csv",
    "behavior_http_mvc_v1.csv",
    "behavior_ef_v1.csv"
]

dataframes = []

for filename in result_files:

    path = EVALUATION_DIR / filename

    assert path.exists(), f"Missing file: {path}"

    dataframes.append(
        pd.read_csv(path)
    )


behavior_results = pd.concat(
    dataframes,
    ignore_index=True
)


# ============================================================
# 2. Normalize Boolean Values
# ============================================================

def to_boolean(value):

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    return str(value).strip().lower() == "true"


behavior_results["passed"] = (
    behavior_results["passed"]
    .map(to_boolean)
)


# ============================================================
# 3. Validate Evaluation Records
# ============================================================

MODEL_NAMES = [
    "base_codellama",
    "optuna_best",
    "mc_dropout"
]

test_ids = list(
    heldout_test_dataset["migration_id"]
)

assert len(behavior_results) == 30

assert not behavior_results.duplicated(
    ["model", "migration_id"]
).any()


for model_name in MODEL_NAMES:

    model_cases = set(
        behavior_results.loc[
            behavior_results["model"] == model_name,
            "migration_id"
        ]
    )

    assert model_cases == set(test_ids)


print("--- Evaluation Data Validation ---")
print("Models:", len(MODEL_NAMES))
print("Migration scenarios:", len(test_ids))
print("Total evaluation records:", len(behavior_results))
print("Data integrity: PASS")


# ============================================================
# 4. Create Paired Results Matrix
# ============================================================

paired_results = behavior_results.pivot(
    index="migration_id",
    columns="model",
    values="passed"
)

paired_results = paired_results.loc[
    test_ids,
    MODEL_NAMES
].astype(int)


print("\n--- Per-Scenario Behavioral Results ---")
print(paired_results.to_string())


# ============================================================
# 5. Model Accuracy
# ============================================================

accuracy = paired_results.mean(axis=0)

print("\n--- Behavioral Accuracy ---")

for model_name in MODEL_NAMES:

    print(
        f"{model_name}: "
        f"{accuracy[model_name]:.4f}"
    )


# ============================================================
# 6. Paired Bootstrap Function
# ============================================================

def paired_bootstrap_ci(
    model_a,
    model_b,
    n_bootstrap=20000,
    confidence_level=0.95,
    seed=42
):

    a = np.asarray(
        model_a,
        dtype=np.float64
    )

    b = np.asarray(
        model_b,
        dtype=np.float64
    )

    assert a.shape == b.shape

    n = len(a)

    differences = a - b

    observed_difference = differences.mean()

    rng = np.random.default_rng(seed)

    # Same resampled task indices for both models.
    sampled_indices = rng.integers(
        low=0,
        high=n,
        size=(n_bootstrap, n)
    )

    bootstrap_differences = (
        differences[sampled_indices]
        .mean(axis=1)
    )

    alpha = 1.0 - confidence_level

    ci_lower, ci_upper = np.quantile(
        bootstrap_differences,
        [
            alpha / 2,
            1 - alpha / 2
        ]
    )

    wins = int(
        np.sum(differences > 0)
    )

    losses = int(
        np.sum(differences < 0)
    )

    ties = int(
        np.sum(differences == 0)
    )

    return {
        "observed_difference": observed_difference,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "wins": wins,
        "losses": losses,
        "ties": ties
    }


# ============================================================
# 7. Model Comparisons
# ============================================================

comparisons = [
    ("optuna_best", "base_codellama"),
    ("mc_dropout", "base_codellama"),
    ("mc_dropout", "optuna_best")
]

bootstrap_results = []


print("\n" + "=" * 65)
print("PAIRED BOOTSTRAP RESULTS")
print("=" * 65)


for model_a, model_b in comparisons:

    result = paired_bootstrap_ci(
        paired_results[model_a].to_numpy(),
        paired_results[model_b].to_numpy(),
        n_bootstrap=20000,
        confidence_level=0.95,
        seed=42
    )

    bootstrap_results.append({
        "model_a": model_a,
        "model_b": model_b,
        **result
    })

    print(f"\nComparison: {model_a} - {model_b}")

    print(
        f"Observed difference: "
        f"{result['observed_difference']:.4f}"
    )

    print(
        f"95% Bootstrap CI: "
        f"[{result['ci_lower']:.4f}, "
        f"{result['ci_upper']:.4f}]"
    )

    print(
        f"Wins: {result['wins']} | "
        f"Losses: {result['losses']} | "
        f"Ties: {result['ties']}"
    )


# ============================================================
# 8. Save Experiment Results
# ============================================================

bootstrap_df = pd.DataFrame(
    bootstrap_results
)

accuracy_path = (
    EVALUATION_DIR
    / "behavioral_accuracy_v1.csv"
)

bootstrap_path = (
    EVALUATION_DIR
    / "paired_bootstrap_v1.csv"
)

paired_results.to_csv(
    accuracy_path
)

bootstrap_df.to_csv(
    bootstrap_path,
    index=False
)


print("\n--- Saved Evaluation Artifacts ---")

print(accuracy_path)
print(bootstrap_path)

print("\nPaired bootstrap evaluation: PASS")

# Conclusion

## CodeMigrate AI - CodeLlama-7B QLoRA Pilot Experiment

This notebook investigated whether QLoRA fine-tuning could improve
CodeLlama-7B-Instruct for migrating legacy .NET Framework C# code
to modern .NET.

The experiment covered model preparation, supervised fine-tuning,
hyperparameter optimization, uncertainty estimation, code generation,
compilation testing, behavioral testing, and statistical evaluation.

### 1. Fine-Tuning and Hyperparameter Optimization

CodeLlama-7B-Instruct was loaded using 4-bit quantization and
fine-tuned with QLoRA.

Optuna was used to evaluate eight hyperparameter configurations,
including LoRA rank, scaling factor, dropout, learning rate,
weight decay, warmup ratio, and learning-rate scheduler.

The best configuration achieved:

- LoRA rank: 16
- LoRA alpha: 16
- Learning rate: approximately 1.90e-4
- Weight decay: 0.06
- LoRA dropout: 0.0
- Scheduler: cosine

The best validation loss was approximately 0.214, compared with
0.567 for the original CodeLlama model.

This represents an approximately 62.3% reduction in validation loss
on the small pilot validation set.

### 2. Independent Code Generation Evaluation

Three model configurations were evaluated:

1. Original CodeLlama-7B-Instruct
2. Optuna-optimized QLoRA adapter
3. QLoRA adapter trained with LoRA dropout

The evaluation included ten migration scenarios covering:

- Configuration management
- HTTP context migration
- ASP.NET MVC migration
- Entity Framework modernization

The fine-tuned models consistently produced code-only responses,
whereas the original model frequently included explanations or
Markdown formatting.

However, improved output formatting did not consistently translate
into correct program behavior.

### 3. Compilation and Behavioral Correctness

Compilation accuracy:

| Model | Compilation Accuracy |
|---|---:|
| Base CodeLlama | 60% |
| Optuna QLoRA | 70% |
| MC Dropout QLoRA | 70% |

Behavioral accuracy:

| Model | Behavioral Accuracy |
|---|---:|
| Base CodeLlama | 60% |
| Optuna QLoRA | 40% |
| MC Dropout QLoRA | 40% |

The fine-tuned models improved compilation accuracy slightly but
performed worse on behavioral correctness in this evaluation.

Observed failure patterns included:

- Changing existing method names
- Incorrect configuration API substitutions
- Missing ASP.NET Core namespace imports
- Incorrect Entity Framework query migrations
- Changes to existing method contracts

These findings indicate that validation loss alone is insufficient
for selecting a code migration model.

### 4. Statistical Evaluation

Paired bootstrap resampling was applied to the ten migration
scenarios.

For Optuna QLoRA versus the original model:

- Observed behavioral accuracy difference: -0.20
- 95% bootstrap confidence interval: [-0.50, 0.00]

The confidence interval includes zero.

Therefore, the available evaluation does not establish a
statistically reliable improvement in behavioral correctness.

The small number of test scenarios also limits the precision
and generalizability of these estimates.

### 5. Monte Carlo Dropout

Monte Carlo Dropout was investigated using repeated stochastic
forward passes.

The experiment estimated:

- Predictive uncertainty
- Reference-token negative log-likelihood
- Predictive entropy
- Epistemic uncertainty through mutual information

The measured epistemic uncertainty was small for the examined
example.

These measurements provide an initial uncertainty analysis but
do not establish that the model's generated C# code is
functionally correct.

### 6. Limitations

The pilot experiment had several limitations:

- Only 29 training examples were available.
- The training data was heavily concentrated on configuration migration.
- Hyperparameter search used only eight trials.
- Each trial involved a short training schedule.
- Behavioral evaluation included only ten scenarios.
- The evaluation did not cover complete enterprise applications.

Furthermore, several evaluation failures were inspected during
development. A new untouched test set is therefore required
for the next training experiment.

### 7. Next Experiment

The next experiment will retain CodeLlama-7B-Instruct and focus
on improving the training data.

Planned improvements include:

1. Collecting publicly licensed legacy and modern .NET code.
2. Constructing reviewed migration pairs at method and class level.
3. Balancing examples across migration categories.
4. Preserving method signatures, configuration keys, and business logic.
5. Validating generated targets through compilation and behavioral tests.
6. Preventing repository-level and near-duplicate data leakage.
7. Repeating QLoRA fine-tuning and Optuna optimization.
8. Evaluating the new model on a fresh, unseen test set.

### Final Assessment

The pilot demonstrated that QLoRA can adapt CodeLlama-7B to a
specific output format using limited training data.

However, the experiment did not demonstrate an improvement in
functional migration correctness.

The next phase will prioritize dataset quality, behavioral
preservation, and more representative evaluation.